In [1]:
import pandas as pd
import numpy as np
import requests
from bs4 import BeautifulSoup
import re
from urllib.parse import urljoin
from io import StringIO
import os
from datetime import datetime, date
import time
from pathlib import Path

In [2]:
from datetime import datetime

In [3]:
import certifi
import ssl
context = ssl.create_default_context(cafile=certifi.where())
context = ssl._create_unverified_context()

In [4]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, ElementClickInterceptedException

In [5]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', 2000)

Delete empty files

In [13]:
files = os.listdir("./sources/fftri_t2_area/")
for f_ in files:
    df_ = pd.read_csv(f"./sources/fftri_t2_area/{f_}")
    if len(df_.index) == 0:
        os.remove(f"./sources/fftri_t2_area/{f_}")

In [25]:
def list_files_folder(folder, path_len=4):
    root = Path(f"./sources/{folder}/")
    output = {}
    for i in [f'./{p}' for p in root.rglob("*") if p.is_file()]:
        if i not in [f"./sources/{folder}/urls.csv", f"./sources/{folder}/.DS_Store"] and len(i.split("/")) == path_len:
            if i.split("/")[-2] in output:
                output[i.split("/")[-2]].append(i)
            else:
                output[i.split("/")[-2]] = [i]
    return output

input_files = list_files_folder("fftri_t2_area") | list_files_folder("prolivesport", path_len=5)

In [26]:
for target in input_files:
    print(target)

fftri_t2_area
prolivesport_51
prolivesport_56
prolivesport_60
prolivesport_58
prolivesport_33
prolivesport_34
prolivesport_59
prolivesport_57
prolivesport_50
prolivesport_35
prolivesport_32
prolivesport_10
prolivesport_17
prolivesport_28
prolivesport_21
prolivesport_26
prolivesport_19
prolivesport_8
prolivesport_6
prolivesport_43
prolivesport_1
prolivesport_44
prolivesport_27
prolivesport_18
prolivesport_20
prolivesport_16
prolivesport_29
prolivesport_11
prolivesport_0
prolivesport_45
prolivesport_7
prolivesport_42
prolivesport_9
prolivesport_37
prolivesport_30
prolivesport_39
prolivesport_55
prolivesport_52
prolivesport_38
prolivesport_31
prolivesport_36
prolivesport_53
prolivesport_54
prolivesport_49
prolivesport_2
prolivesport_47
prolivesport_5
prolivesport_40
prolivesport_14
prolivesport_13
prolivesport_25
prolivesport_22
prolivesport_4
prolivesport_41
prolivesport_3
prolivesport_46
prolivesport_48
prolivesport_23
prolivesport_24
prolivesport_12
prolivesport_15


In [2]:
!pip3 install PyPDF2

Defaulting to user installation because normal site-packages is not writeable
     |████████████████████████████████| 232 kB 5.3 MB/s eta 0:00:01
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.


In [1]:
import PyPDF2

In [2]:
reader = PyPDF2.PdfReader('20230625_IRONMAN_France___IRONMAN_70.3_Nice_IRONMAN_Age_Groups_OverallResults_20230626145139.pdf')

In [3]:
print(len(reader.pages))

60


In [4]:
print(reader.pages[0].extract_text())

IRONMAN France & IRONMAN 70.3 Nice
June 25, 2023
IRONMAN Age Groups - Overall Results
PosStart # Athlete Gender Division Country
 T1
 T2
 Time
1 1669 Anthony Roux M M35-39
       - FRA 01:01:18 00:05:04 04:31:05 00:03:59 03:10:37 08:52:00
2 1236 Alex Woodman M M30-34
       - GBR 00:48:02 00:03:45 04:59:20 00:03:30 02:59:33 08:54:08
3 127 Gonzalo GARVEY M M30-34
       - ESP 00:53:56 00:04:36 05:01:22 00:02:50 02:57:38 09:00:20
4 710 Cedric Labadens M M35-39
       - FRA 00:55:37 00:04:34 05:15:31 00:03:44 02:53:51 09:13:15
5 562 Morgan FALSE M M25-29
       - FRA 00:55:03 00:05:46 05:13:54 00:04:32 03:02:43 09:21:55
6 205 Julien Viennot M M35-39
       - FRA 00:59:49 00:06:34 05:00:51 00:04:05 03:14:42 09:25:58
7 210 Benjamin Bouchet M M35-39
       - FRA 00:55:21 00:05:31 05:21:17 00:03:27 03:01:59 09:27:32
8 1591 Paulin Philippe M M18-24
       - FRA 00:48:13 00:04:39 05:02:12 00:02:32 03:32:20 09:29:54
9 327 Aurelio GAUTHIER M M25-29
       - FRA 00:59:48 00:05:26 05:17:39 00:03:45

# Data from PDF IronMan Nice AG

In [5]:
rows_per_person = 2

In [6]:
reader = PyPDF2.PdfReader('20230625_IRONMAN_France___IRONMAN_70.3_Nice_IRONMAN_Age_Groups_OverallResults_20230626145139.pdf')

In [7]:
output = list()
for index_page in range(len(reader.pages)):
    text_page = reader.pages[index_page].extract_text().split("\n")[7:-1]
    chunks = [text_page[x:x+rows_per_person] for x in range(0, len(text_page), rows_per_person)]

    for player_data in chunks:
        overall_position = player_data[0]
        person = " ".join(player_data[0].split(" ")[2:-2])
        final_time = player_data[1].split(" ")[-1]
        nation = player_data[1].split(" ")[8]
        output.append([person,nation,final_time])

In [8]:
print(output[:3])

[['Anthony Roux', 'FRA', '08:52:00'], ['Alex Woodman', 'GBR', '08:54:08'], ['Gonzalo GARVEY', 'ESP', '09:00:20']]


# Data from PDF EmbrunmanXXL

In [9]:
rows_per_person = 3

In [10]:
reader = PyPDF2.PdfReader('01RsultatsenligneClassementGnral.pdf')

In [11]:
output = list()
for index_page in range(len(reader.pages)):
    text_page = reader.pages[index_page].extract_text().split("\n")[:-5]
    chunks = [text_page[x:x+rows_per_person] for x in range(0, len(text_page), rows_per_person)]

    for player_data in chunks:
        nation = player_data[1].split(" ")[-1]
        person = player_data[1].replace(nation,"")[:-1]
        final_time = player_data[2].split(" ")[-1]
        output.append([person,nation,final_time])

In [12]:
print(output[:3])

[['CHEVALIER Leon', 'FRA', '9:28:18'], ['DIEMUNSCH Etienne', 'FRA', '9:38:07'], ['VISTICA Andrej', 'CRO', '9:38:27']]


# DATA OF PROLIVESPORT

In [41]:
# récupération de toutes les compétitions de Prolivesport

In [5]:
base_url = "https://www.prolivesport.fr"
url = base_url + "/resultat-classement/"

resp = requests.get(url)
resp.raise_for_status()

soup = BeautifulSoup(resp.text, "html.parser")

# trouver le tableau principal
table = soup.find("table")

rows = []
for tr in table.find_all("tr"):
    tds = tr.find_all("td")
    if len(tds) != 5:
        continue  # skip header or weird rows

    date = tds[0].get_text(strip=True)
    epreuve = tds[1].get_text(strip=True)
    lieu = tds[2].get_text(strip=True)
    categorie = tds[3].get_text(strip=True)

    # cellule résultats : texte + href
    a = tds[4].find("a")
    if a is not None:
        result_text = a.get_text(strip=True)
        href = a.get("href")
        # gérer les liens relatifs
        if href and href.startswith("/"):
            href = base_url + href
    else:
        result_text = tds[4].get_text(strip=True)
        href = None

    rows.append({
        "date": date,
        "epreuve": epreuve,
        "lieu": lieu,
        "categorie": categorie,
        "resultat_label": result_text,
        "resultat_url": href,
    })

df = pd.DataFrame(rows)

AttributeError: 'NoneType' object has no attribute 'find_all'

In [14]:
df.head()

,date,epreuve,lieu,categorie,resultat_label,resultat_url
0,07/12/2025,Bike and Run Halluin 2025,HalluinNord / Hauts-de-France,FFTRI - Bike & Run,LIVE,https://www.prolivesport.fr/index.php?chap=event&sub=liveV3&eventId=1139
1,05/12/2025,La Noctambule 2025 - Thème Harry PotterTéléthon,La GorgueNord / Hauts-de-France,C.O. -> Course d'Orientation / Epreuve familiale,Résultats,https://docs.google.com/spreadsheets/d/1zGGxcCSnr-zxMC7G-t2Ul4CYB5NPUnhhwVlied-z7r0/edit?usp=sharing
2,21/11/2025,#STOP aux Violences Faites aux Femmes 2025,LilleNord / Hauts-de-France,FFA Athlétisme => Running & Epreuve solidaire,LIVE,https://www.prolivesport.fr/index.php?chap=event&sub=liveV3&eventId=1063
3,02/11/2025,La pévèle Trail 2025,FretinNord / Hauts-de-France,FFA Athlétisme => Running trail,LIVE,https://www.prolivesport.fr/index.php?chap=event&sub=liveV3&eventId=1110
4,02/11/2025,Foulées lesquinoises 2025,LesquinNord / Hauts-de-France,FFA Athlétisme => Running Route,LIVE,https://www.prolivesport.fr/index.php?chap=event&sub=liveV3&eventId=1133


In [24]:
import json

In [38]:
def call_api_prolivesport(url_target):
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
            "(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
        ),
        "Referer": "https://www.prolivesport.fr/",
        "access-token": "AUTH_PLSWS_V2",
    }
    
    try:
        response = requests.get(url_target, headers=headers)
        response.raise_for_status()
        
        data = response.json()
        return data
    
    except requests.exceptions.RequestException as e:
        print(f"Erreur réseau : {e}")
        return None

In [30]:
global_page_result = "https://api.prolivesport.fr/apiws/event/result/"
races = call_api_prolivesport(global_page_result)

extracted_data = []
for r in races["result"]:
    extracted_data.append({"id":r["eventId"], 
                           "name":r["eventName"], 
                           "date":r["eventDateStart"], 
                           "categorieevent":r["categoryName"], 
                           "categorievar":r["categoryVar"],
                          "zipcode":r["zipcode"],
                          "city":r["city"],
                          "department":r["department"],
                          "country":r["country"]})

In [33]:
for e in extracted_data:
    id_event = e["id"]
    break

In [37]:
extracted_data[0]

{'id': '1258',
 'name': "Trail de la forêt d'Eperlecques 2026",
 'date': '2026-11-07 00:00:00',
 'categorieevent': 'Course à pied - Running',
 'categorievar': 'course-a-pied-running',
 'zipcode': '62910',
 'city': 'Éperlecques',
 'department': '62 - Pas-de-Calais',
 'country': 'France'}

In [55]:
url = "https://api.prolivesport.fr/apiws/result/raceList/1262/"
racelist = call_api_prolivesport(url)
racelist

{'success': True,
 'message': 'result',
 'result': [{'race': 'Open', '0': 'Open', 'distance': '25.75', '1': '25.75'},
  {'race': 'PTS2-5-F', '0': 'PTS2-5-F', 'distance': '25.75', '1': '25.75'},
  {'race': 'PTS2-5-M', '0': 'PTS2-5-M', 'distance': '25.75', '1': '25.75'},
  {'race': 'PTS7-8', '0': 'PTS7-8', 'distance': '25.75', '1': '25.75'},
  {'race': 'PTVI', '0': 'PTVI', 'distance': '25.75', '1': '25.75'},
  {'race': 'PTWC', '0': 'PTWC', 'distance': '25.75', '1': '25.75'}]}

In [49]:
specific_page_result = "https://api.prolivesport.fr/apiws/result/indiv/1262/Open/"
data = call_api_prolivesport(specific_page_result)

In [53]:
athletes = []
for d in data["result"]:
    athletes.append({"lastname":d["lastname"],
                    "firstname":d["firstname"],
                    "sex":d["sex"],
                    "club":d["club"],
                     "category":d["category"],
                    "categoryRef":d["categoryRef"],
                    "time":d["time"],
                    "rankSex":d["rankSex"],
                    "rankCat":d["rankCat"],
                    "nation":d["nation"],
                    "birthDate":d["birthDate"],
                    "swim":d["timeT1"],
                    "t1":d["timeT2"],
                     "bike":d["timeT3"],
                     "t2":d["timeT4"],
                     "run":d["timeT5"],
                    })

In [54]:
athletes

[{'lastname': 'GOETZ',
  'firstname': 'JACKY',
  'sex': 'M',
  'club': 'SAVERNE ENDURANCE TRIATHLON',
  'category': 'Open',
  'categoryRef': 'O',
  'time': '01:09:35',
  'rankSex': '1',
  'rankCat': '1',
  'nation': 'FRA',
  'birthDate': '0000-00-00',
  'swim': '00:14:36',
  't1': '00:02:00',
  'bike': '00:33:10',
  't2': '00:01:31',
  'run': '00:18:20'},
 {'lastname': 'BAHI',
  'firstname': 'KAMEL',
  'sex': 'M',
  'club': 'STADE FRANCAIS',
  'category': 'Open',
  'categoryRef': 'O',
  'time': '01:14:45',
  'rankSex': '2',
  'rankCat': '2',
  'nation': 'FRA',
  'birthDate': '0000-00-00',
  'swim': '00:16:30',
  't1': '00:02:24',
  'bike': '00:35:57',
  't2': '00:01:40',
  'run': '00:18:16'},
 {'lastname': 'BERTHE',
  'firstname': 'LAURENT',
  'sex': 'M',
  'club': 'LE CLUB TRIATHLON',
  'category': 'Open',
  'categoryRef': 'O',
  'time': '01:17:52',
  'rankSex': '3',
  'rankCat': '3',
  'nation': 'FRA',
  'birthDate': '0000-00-00',
  'swim': '00:15:47',
  't1': '00:02:09',
  'bike': '

In [56]:
data["result"][0]

{'liveId': '401542479',
 '0': '401542479',
 'athleteId': '',
 '1': '',
 'eventId': '1262',
 '2': '1262',
 'race': 'Open',
 '3': 'Open',
 'pointName': 'Finish',
 '4': 'Finish',
 'number': '907',
 '5': '907',
 'sex': 'M',
 '6': 'M',
 'lastname': 'GOETZ',
 '7': 'GOETZ',
 'firstname': 'JACKY',
 '8': 'JACKY',
 'club': 'SAVERNE ENDURANCE TRIATHLON',
 '9': 'SAVERNE ENDURANCE TRIATHLON',
 'category': 'Open',
 '10': 'Open',
 'categoryRef': 'O',
 '11': 'O',
 'time': '01:09:35',
 '12': '01:09:35',
 'rank': '1',
 '13': '1',
 'rankSex': '1',
 '14': '1',
 'rankCat': '1',
 '15': '1',
 'liveRank': '1',
 '16': '1',
 'liveTime': None,
 '17': None,
 'distance': '25.75',
 '18': '25.75',
 'organisme': 'HC',
 '19': 'HC',
 'sponsor': '',
 '20': '',
 'nation': 'FRA',
 '21': 'FRA',
 'email': '',
 '22': '',
 'mobile': '',
 '23': '',
 'birthDateYear': '1979',
 '24': '1979',
 'birthDate': '0000-00-00',
 '25': '0000-00-00',
 'birthDateLocal': '//1979',
 '26': '//1979',
 'licence': 'C25118C10160218MV2FRA',
 '27': '

In [24]:
df[df["categorie"]=='FF Triathlon => Triathlon']["resultat_url"].to_list()

['https://www.prolivesport.fr/index.php?chap=event&sub=liveV3&eventId=1108',
 'https://www.prolivesport.fr/index.php?chap=event&sub=liveV3&eventId=1129',
 'https://www.prolivesport.fr/index.php?chap=event&sub=liveV3&eventId=1128',
 'https://www.prolivesport.fr/index.php?chap=event&sub=liveV3&eventId=1128',
 'https://www.prolivesport.fr/index.php?chap=event&sub=liveV3&eventId=1082',
 'https://www.prolivesport.fr/index.php?chap=event&sub=liveV3&eventId=1080',
 'https://www.prolivesport.fr/index.php?chap=event&sub=liveV3&eventId=1082',
 'https://www.prolivesport.fr/index.php?chap=event&sub=liveV3&eventId=1124',
 'https://www.prolivesport.fr/index.php?chap=event&sub=liveV3&eventId=1124',
 'https://www.prolivesport.fr/index.php?chap=event&sub=liveV3&eventId=1102',
 'https://www.prolivesport.fr/index.php?chap=event&sub=liveV3&eventId=1079',
 'https://www.prolivesport.fr/index.php?chap=event&sub=liveV3&eventId=1079',
 'https://www.prolivesport.fr/index.php?chap=event&sub=liveV3&eventId=1094']

In [19]:
triathlon_events = df[df["categorie"]=='FF Triathlon => Triathlon']
for index, row in triathlon_events.iterrows():
    print(index)
    print(row)
    break

12
date                                                                            05/10/2025
epreuve                                                     Cross Duathlon du Haricot 2025
lieu                                                    ZegerscappelNord / Hauts-de-France
categorie                                                        FF Triathlon => Triathlon
resultat_label                                                                        LIVE
resultat_url      https://www.prolivesport.fr/index.php?chap=event&sub=liveV3&eventId=1108
Name: 12, dtype: object


In [40]:
# obtenir la liste des épreuves sur une compétition

In [7]:
base_url = "https://www.prolivesport.fr"
url = "https://www.prolivesport.fr/index.php?chap=event&sub=liveV3&eventId=1128"
url = "https://www.prolivesport.fr/index.php?chap=event&sub=liveV3&eventId=1079"

resp = requests.get(url)
resp.raise_for_status()

soup = BeautifulSoup(resp.text, "html.parser")

items = []
# on cherche les <a> qui sont dans la zone "Sélection de la compétition"
for a in soup.find_all("a"):
    txt = a.get_text(strip=True)
    if not txt:
        continue

    # on filtre pour ne garder que les compétitions probables
    if not re.search(r"\d", txt):# and txt not in ("M", "XS"):
        continue

    # texte du lien + éventuelle distance dans le texte ou à côté
    name = txt

    # distance éventuelle dans le même texte: "10-13 ans (5.9 Km)"
    m = re.search(r"\(([^)]+)\)", txt)
    if m:
        distance = m.group(1).strip()
    else:
        # sinon on regarde le texte juste après le lien "(5.9 Km)"
        sib_text = a.next_sibling
        if sib_text:
            raw = sib_text.strip()
            m2 = re.match(r"\(([^)]+)\)", raw)
            distance = m2.group(1).strip() if m2 else ""
        else:
            distance = ""

    href = a.get("href")
    if href:
        href = urljoin(base_url, href)

    items.append({
        "name": name,
        "distance": distance,
        "url": href,
    })

for it in items:
    print(it)

{'name': 'M(49.5 Km)', 'distance': '49.5 Km', 'url': 'https://www.prolivesport.fr/index.php?chap=event&sub=liveV3&eventId=1079&race=M'}
{'name': 'S(24.75 Km)', 'distance': '24.75 Km', 'url': 'https://www.prolivesport.fr/index.php?chap=event&sub=liveV3&eventId=1079&race=S'}
{'name': 'XS(13.9 Km)', 'distance': '13.9 Km', 'url': 'https://www.prolivesport.fr/index.php?chap=event&sub=liveV3&eventId=1079&race=XS'}


In [39]:
# obtenir les résultats d'une épreuve

In [8]:
url = "https://www.prolivesport.fr/index.php?chap=event&sub=liveV3&eventId=1128&race=S"
url = "https://www.prolivesport.fr/index.php?chap=event&sub=liveV3&eventId=1124&race=Triacross%20S"
results_final = pd.DataFrame()
for i in range(1,100):
    html = requests.get(url+"&page="+str(i),verify=False)
    tables = pd.read_html(html.text)
    try:
        results_final = pd.concat([results_final, tables[1]], ignore_index=True)
    except:
        break

/Users/maclv/Work/dev/chrono/lib/python3.14/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.prolivesport.fr'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/var/folders/hm/d934n_zn2r14w19nr22fmj_00000gn/T/ipykernel_8987/1090414735.py:6: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(html.text)
/Users/maclv/Work/dev/chrono/lib/python3.14/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.prolivesport.fr'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/var/folders/hm/d934n_zn2r14w19nr22fmj_00000gn/T/

In [9]:
results_final.head()

,Clt,N°,Athlète,Nat.,Club,Temps,Sexe,Cat,Swim,#1,Bike,#2,Run,Unnamed: 13
0,1,278,THIREAU Amaury,NaN,TUVB TRIATHLON,00:44:52,M (1e),Junior (1e),00:07:49 (1e),00:00:43 (1e),00:20:07 (2e),00:00:38 (3e),00:15:37 (1e),NaN
1,2,399,JOLY Romain,NaN,J3 SPORTS AMILLY SECTION TRIATHLON,00:47:30,M (2e),Senior (1e),00:09:02 (8e),00:01:16 (8e),00:19:31 (1e),00:00:44 (9e),00:16:59 (5e),NaN
2,3,379,KHUAT DUY Maxime,NaN,US PALAISEAU TRIATHLON,00:47:37,M (3e),Senior (2e),00:08:32 (4e),00:01:07 (5e),00:21:22 (7e),00:00:35 (2e),00:16:02 (2e),NaN
3,4,312,JAULIN Leo,NaN,E.C. SARTROUVILLE,00:47:55,M (4e),Cadet (1e),00:08:40 (5e),00:00:56 (2e),00:21:17 (6e),00:00:32 (1e),00:16:32 (4e),NaN
4,5,346,ASSELBOURG Valentin,NaN,NaN,00:49:59,M (5e),Cadet (2e),00:07:53 (2e),00:01:03 (3e),00:20:22 (3e),00:00:59 (28e),00:19:45 (25e),NaN


In [11]:
url = "https://www.prolivesport.fr/index.php?chap=event&sub=liveV3&eventId=1124&race=Triacross%20S"
i = 1
html = requests.get(url+"&page="+str(i),verify=False)
tables = pd.read_html(html.text)

/Users/maclv/Work/dev/chrono/lib/python3.14/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.prolivesport.fr'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/var/folders/hm/d934n_zn2r14w19nr22fmj_00000gn/T/ipykernel_8987/1207769066.py:4: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(html.text)


In [19]:
result_tmp = pd.DataFrame()
for table in tables[1:]:
    result_tmp = pd.concat([result_tmp, table], ignore_index=True)

In [20]:
result_tmp

,Clt,N°,Athlète,Nat.,Club,Temps,Sexe,Cat,Swim,#1,Bike,#2,Run,Unnamed: 13
0,1,278,THIREAU Amaury,NaN,TUVB TRIATHLON,00:44:52,M (1e),Junior (1e),00:07:49 (1e),00:00:43 (1e),00:20:07 (2e),00:00:38 (3e),00:15:37 (1e),NaN
1,2,399,JOLY Romain,NaN,J3 SPORTS AMILLY SECTION TRIATHLON,00:47:30,M (2e),Senior (1e),00:09:02 (8e),00:01:16 (8e),00:19:31 (1e),00:00:44 (9e),00:16:59 (5e),NaN
2,3,379,KHUAT DUY Maxime,NaN,US PALAISEAU TRIATHLON,00:47:37,M (3e),Senior (2e),00:08:32 (4e),00:01:07 (5e),00:21:22 (7e),00:00:35 (2e),00:16:02 (2e),NaN
3,4,312,JAULIN Leo,NaN,E.C. SARTROUVILLE,00:47:55,M (4e),Cadet (1e),00:08:40 (5e),00:00:56 (2e),00:21:17 (6e),00:00:32 (1e),00:16:32 (4e),NaN
4,5,346,ASSELBOURG Valentin,NaN,NaN,00:49:59,M (5e),Cadet (2e),00:07:53 (2e),00:01:03 (3e),00:20:22 (3e),00:00:59 (28e),00:19:45 (25e),NaN
5,6,350,BRUNET Axel,NaN,NaN,00:50:16,M (6e),Junior (2e),00:10:25 (20e),00:01:42 (22e),00:21:12 (5e),00:00:44 (10e),00:16:15 (3e),NaN
6,7,394,GAULUPEAU Nathan,NaN,NaN,00:51:03,M (7e),Cadet (3e),00:08:02 (3e),00:01:05 (4e),00:23:02 (18e),00:00:39 (4e),00:18:17 (10e),NaN
7,8,390,BUREL Baptiste,NaN,TUVB TRIATHLON,00:51:09,M (8e),Cadet (4e),00:08:59 (7e),00:01:07 (6e),00:22:31 (16e),00:00:40 (5e),00:17:54 (9e),NaN
8,9,308,AVIGNON Flavien,NaN,NaN,00:51:46,M (9e),Senior (3e),00:10:22 (19e),00:01:34 (19e),00:21:51 (8e),00:00:47 (15e),00:17:14 (7e),NaN
9,10,347,ASSELBOURG Benjamin,NaN,NaN,00:52:03,M (10e),Master (1e),00:10:40 (24e),00:01:16 (9e),00:20:40 (4e),00:00:54 (21e),00:18:34 (13e),NaN


# Récupération des données de Protiming

In [7]:
url = "https://protiming.fr/results"
resp = requests.get(url)
resp.raise_for_status()
soup = BeautifulSoup(resp.text, "html.parser")
content = soup.find("div", id="contentAjax")
if not content:
    raise RuntimeError("div#contentAjax introuvable")
rows = content.find_all("div", class_="row")

text_rows = []
for row in rows:
    text_rows.append(row.get_text(separator="\n", strip=True))

RuntimeError: div#contentAjax introuvable

In [105]:
cleaned_events = []
#prendre 1 element sur 2
for event in text_rows[1::2]:
    event_name, event_place = event.split("\n")
    cleaned_event_place = re.sub(r'\s+', ' ', event_place.strip())
    cleaned_events.append({"event_name": event_name, "location": cleaned_event_place})

unique_events = []
for d in cleaned_events:
    if d not in unique_events:
        unique_events.append(d)

In [108]:
unique_events[:4]

[{'event_name': 'Trail des Pères Noël', 'location': 'Créon (33)'},
 {'event_name': 'Cross SDIS 28 – Voves',
  'location': 'Les villages Vovéens (28)'},
 {'event_name': "Ruche Run'Nature", 'location': 'Nouan-le-Fuzelier (41)'},
 {'event_name': 'La Course des Pères Noël', 'location': 'Cissac-Médoc (33)'}]

In [99]:
url = "https://protiming.fr/Results/liste"
resp = requests.get(url)
resp.raise_for_status()

soup = BeautifulSoup(resp.text, "html.parser")

# cibler le conteneur principal
content_ajax = soup.find("div", id="contentAjax")
if not content_ajax:
    print("div#contentAjax non trouvé")
    exit()

# tous les liens à l'intérieur
hrefs = []
for a in content_ajax.find_all("a", href=True):
    full_url = urljoin(url, a["href"])
    text_content = a.get_text(strip=True)
    if "edition" in text_content.lower():
        hrefs.append({"full_url": full_url})
hrefs[:4]

[{'full_url': 'https://protiming.fr/Results/running/7906-Trail-des-Peres-Noel/8376'},
 {'full_url': 'https://protiming.fr/Results/running/7890-Cross-SDIS-28-Voves/9685'},
 {'full_url': 'https://protiming.fr/Results/running/8009-Ruche-Run-Nature/6396'},
 {'full_url': 'https://protiming.fr/Results/running/7666-La-Course-des-Peres-Noel/8186'}]

In [110]:
for name_loc_dict, url_dict in zip(unique_events, hrefs):
    print(name_loc_dict | url_dict)
    break

{'event_name': 'Trail des Pères Noël', 'location': 'Créon (33)', 'full_url': 'https://protiming.fr/Results/running/7906-Trail-des-Peres-Noel/8376'}


In [33]:
base_url = "https://protiming.fr"
url = f"{base_url}/results?page=1"
resp = requests.get(url)
resp.raise_for_status()
soup = BeautifulSoup(resp.text, "html.parser")

In [38]:
nav_pagination = soup.find("nav", attrs={"aria-label": "pagination"})
li_elements = nav_pagination.find_all("li") if nav_pagination else []

# Récupérer les textes non vides (['1', '…', '196', '197'])
textes = [li.get_text(strip=True) for li in li_elements if li.get_text(strip=True)]

# Récupérer directement le numéro maximum de page (197)
pages = [int(t) for t in textes if t.isdigit()]
derniere_page = max(pages) if pages else 1

print(derniere_page)

In [31]:
rows = []
for a_element in soup.find_all("a", href=True):
    if '/results/' in a_element["href"].lower():
        #print(a_element)
        #print(a_element.text)
        nom_epreuve_element = a_element.find("h3")
        nom_epreuve = nom_epreuve_element.get_text(strip=True) if nom_epreuve_element else None
        spans = [span.get_text(strip=True) for span in a_element.find_all("span")]
        if nom_epreuve and len(spans) == 2:
            rows.append({"name": nom_epreuve, "date":spans[0], "lieu": spans[1], "url": base_url+a_element["href"]})

In [32]:
rows

[{'name': "Trail d'Ecquevilly",
  'date': '15 novembre 2026',
  'lieu': 'Ecquevilly (78)',
  'url': 'https://protiming.fr/results/8579-trail-d-ecquevilly'},
 {'name': 'La Boucle Royale',
  'date': '15 novembre 2026',
  'lieu': 'La Ferté Saint Aubin (45)',
  'url': 'https://protiming.fr/results/8521-la-boucle-royale'},
 {'name': 'Cross SDIS 28 – Margon',
  'date': '15 novembre 2026',
  'lieu': 'Margon (28)',
  'url': 'https://protiming.fr/results/8491-cross-sdis-28-margon'},
 {'name': 'Les Foulées du Dragon',
  'date': '15 novembre 2026',
  'lieu': 'La Chapelle-Saint-Mesmin (45)',
  'url': 'https://protiming.fr/results/8342-les-foulees-du-dragon'},
 {'name': 'Trail en Jalle',
  'date': '15 novembre 2026',
  'lieu': "Saint-Jean-d'Illac (33)",
  'url': 'https://protiming.fr/results/8137-trail-en-jalle'},
 {'name': 'En baskets contre le diabète - Novo',
  'date': '14 novembre 2026',
  'lieu': 'Chartres (28)',
  'url': 'https://protiming.fr/results/8408-en-baskets-contre-le-diabete-novo'},


In [91]:
#OLD

protiming = pd.DataFrame()
for page in range(1,100):
    url = f"https://protiming.fr/Results/liste/page:{page}?url=Results%2Fliste"
    base_url = "https://protiming.fr/"
    
    resp = requests.get(url)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, "html.parser")
    
    events = []
    for a in soup.find_all("a"):
        href = a.get("href")
        name = a.get_text(strip=True)
    
        if not href or not name:
            continue
    
        # on ne garde que les liens de résultats de course
        # adapte le pattern si besoin en regardant le HTML (ex: '/Results/race/...')
        if not re.search(r"/Results?/|/resultat|/race", href, re.I):
            continue
    
        full_url = urljoin(base_url, href)
        events.append({"competition": name, "url": full_url})
    
    df = pd.DataFrame(events).drop_duplicates()
    protiming = pd.concat([protiming, df], ignore_index=True)
    if page==12:
        break

In [94]:
for links in protiming["url"]:
    if links in ["https://protiming.fr/Results/liste", "https://protiming.fr/Results/runnerSearch", "https://protiming.fr/results/liste/reset_filters"]:
        continue
    elif re.search(r"/Results/liste/page:(\d+)\?url=Results%2Fliste", links):
        continue
    elif ('Triathlon' in links) or ('Duathlon' in links):
        print(links)
    else:
        continue

https://protiming.fr/Results/running/7880-Duathlon-de-Villenave-d-Ornon/1730
https://protiming.fr/Results/liste/run%3DDuathlon%20de%20Villenave%20d%27Ornon
https://protiming.fr/Results/running/7835-Triathlon-de-St-Georges-sur-Eure-Chartres-Metropole/637
https://protiming.fr/Results/liste/run%3DTriathlon%20de%20St%20Georges%20sur%20Eure%20Chartres%20M%C3%A9tropole
https://protiming.fr/Results/running/7607-Triathlon-de-l-Etang-du-Puits/4057
https://protiming.fr/Results/liste/run%3DTriathlon%20de%20l%27Etang%20du%20Puits
https://protiming.fr/Results/running/7698-Triathlon-de-Brin-d-Amour/654
https://protiming.fr/Results/liste/run%3DTriathlon%20de%20Brin%20d%27Amour
https://protiming.fr/Results/running/7578-Triathlon-Orleans-Ile-Charlemagne/7971
https://protiming.fr/Results/liste/run%3DTriathlon%20Orl%C3%A9ans%20Ile%20Charlemagne


In [100]:
def extract_last_id(url):
    match = re.search(r'/(\d+)$', url)
    return int(match.group(1)) if match else None

In [104]:
url_test = "https://protiming.fr/Results/liste/run%3DDuathlon%20de%20Villenave%20d%27Ornon"
url_test = "https://protiming.fr/Results/running/7880-Duathlon-de-Villenave-d-Ornon/1730"
extract_last_id(url_test)

1730

In [14]:
from urllib.parse import urlparse

url = "https://protiming.fr/Results/running/7578-Triathlon-Orleans-Ile-Charlemagne/7971"

def extract_race_name(url):
    path = urlparse(url).path
    parts = path.strip("/").split("/")
    slug = parts[2]
    name = slug.split("-", 1)[1]
    name = name.replace("-", " ")
    return name

def create_race_unicity(url):
    path = urlparse(url).path
    parts = path.strip("/").split("/")
    return "_".join(parts[2:])

print(extract_race_name(url))
print(create_race_unicity(url))

Triathlon Orleans Ile Charlemagne
7578-Triathlon-Orleans-Ile-Charlemagne_7971


In [74]:
def extract_date_race(soup):
    pattern = re.compile(r"Édition du\s+([0-9]{2}/[0-9]{2}/[0-9]{4})", re.IGNORECASE)
    date_edition = None
    for elem in soup.find_all(text=pattern):
        m = pattern.search(elem)
        if m:
            date_edition = m.group(1)
            break
    return date_edition

In [99]:
# Récupération des résultats de l'épreuve

In [103]:
url = "https://protiming.fr/Results/running/7880-Duathlon-de-Villenave-d-Ornon/1730"+"/page:1"
html = requests.get(url,verify=False)
soup = BeautifulSoup(html.text, 'html.parser')

/Users/maclv/Work/dev/chrono/lib/python3.12/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'protiming.fr'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


In [75]:
extract_date_race(soup)

/var/folders/hm/d934n_zn2r14w19nr22fmj_00000gn/T/ipykernel_3537/118870718.py:4: DeprecationWarning: The 'text' argument to find()-type methods is deprecated. Use 'string' instead.
  for elem in soup.find_all(text=pattern):


'19/10/2025'

In [27]:
select_element = soup.find("select", {"id": "selectDistanceResult"})

options = [(option['value'], option.text.strip()) for option in select_element.find_all("option")]
print(options)

[('654', 'Triathlon XS'), ('9770', 'Distance Natation'), ('9771', 'Distance Vélo'), ('9772', 'Distance Course à pied')]


In [105]:
#TODO : replace id epreuve par chacun de ses id et boucler sur toutes les pages

In [10]:
base_url = "https://protiming.fr/"
url = "https://protiming.fr/Results/running/7880-Duathlon-de-Villenave-d-Ornon/1730"
path = urlparse(url).path
parts = path.strip("/").split("/")
urljoin(base_url, "/".join(parts[:-1]))

'https://protiming.fr/Results/running/7880-Duathlon-de-Villenave-d-Ornon'

In [97]:
pd.read_html(html.text)

/var/folders/17/m86cb52j74j5j9wdp8r4q8pw0000gn/T/ipykernel_1671/4272687825.py:1: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  pd.read_html(html.text)


[    Position      Pos.                Temps                 Nom - Prénom   Club       Dossard  Catégorie           km/h            CaP1             T1            Velo             T2  CaP2  Unnamed: 13
 0          1  01:08:08     BONALUMI Clement       MATERIEL VELO BORDEAUX  80169   S1M (1 / 7)       24.2   00:17:05 (7)   00:00:44 (28)   00:39:03 (1)   00:00:53 (82)  00:10:26 (15)   NaN          NaN
 1          2  01:08:34         GROLIERE Luc                          NaN  80386  V1M (1 / 14)       24.0   00:16:02 (1)   00:00:34 (11)   00:42:10 (8)   00:00:35 (29)   00:09:15 (2)   NaN          NaN
 2          3  01:09:24       DE WEVER Lucas                          NaN  80176  S3M (1 / 16)       23.7   00:16:44 (4)    00:00:27 (3)   00:42:01 (7)    00:00:24 (5)   00:09:49 (6)   NaN          NaN
 3          4  01:09:43      CHATAIGNIER Teo                          NaN  80391   CAM (1 / 3)       23.6   00:16:15 (3)   00:00:37 (14)   00:42:18 (9)   00:00:29 (11)  00:10:07 (10)   NaN    

In [42]:
url = "https://protiming.fr/results/8205-duathlon-de-villenave-d-ornon"
html = requests.get(url)
soup = BeautifulSoup(html.text, 'html.parser')
input_epreuve = soup.find("input")

if input_epreuve:
    valeur = input_epreuve.get("value")       # "Duathlon S - Individuel"
    placeholder = input_epreuve.get("placeholder") # "Épreuve"
    print(f"{placeholder} : {valeur}")

# Méthode 2 : Si vous avez plusieurs inputs similaires dans la page
for div in soup.find_all("div", attrs={"data-slot": "input-group"}):
    inp = div.find("input", attrs={"data-slot": "input-group-control"})
    if inp:
        print({
            "champ": inp.get("placeholder"),
            "valeur": inp.get("value")
        })

ton-nom@email.fr : None


In [51]:
url = "https://protiming.fr/results/8205-duathlon-de-villenave-d-ornon"

soup = BeautifulSoup(url, "html.parser")

input_element = soup.find(
    "input",
    {"data-slot": "input-group-control"}
)

value = input_element.get("value") if input_element else None

print(value)
# Duathlon S - Individuel

None


/var/folders/hm/d934n_zn2r14w19nr22fmj_00000gn/T/ipykernel_28332/2880567918.py:3: MarkupResemblesLocatorWarning: The input passed in on this line looks more like a URL than HTML or XML.

If you meant to use Beautiful Soup to parse the web page found at a certain URL, then something has gone wrong. You should use an Python package like 'requests' to fetch the content behind the URL. Once you have the content as a string, you can feed that string into Beautiful Soup.

However, if you want to parse some data that happens to look like a URL, then nothing has gone wrong: you are using Beautiful Soup correctly, and this warning is spurious and can be filtered. To make this warning go away, run this code before calling the BeautifulSoup constructor:

    from bs4 import MarkupResemblesLocatorWarning
    import warnings

    warnings.filterwarnings("ignore", category=MarkupResemblesLocatorWarning)
    
  soup = BeautifulSoup(url, "html.parser")


In [60]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

options = webdriver.ChromeOptions()
options.add_argument("--start-maximized")

driver = webdriver.Chrome(options=options)

driver.get(url)




elements = driver.find_elements(
    By.CSS_SELECTOR,
    'input[data-slot="input-group-control"]'
)

print(f"Nombre d'éléments trouvés : {len(elements)}")

for i, element in enumerate(elements):
    print(f"Élément {i + 1}")
    print(f"  Valeur : {element.get_attribute('value')}")
    print(f"  Placeholder : {element.get_attribute('placeholder')}")
    print(f"  ID : {element.get_attribute('id')}")
    print(f"  Type : {element.get_attribute('type')}")

    print(element.get_attribute("outerHTML"))

Nombre d'éléments trouvés : 4
Élément 1
  Valeur : 
  Placeholder : Chercher un participant
  ID : base-ui-_r_1_
  Type : text
<input data-slot="input-group-control" class="h-auto w-full min-w-0 border-grey-400 px-4 py-3.5 text-(length:--text-m) leading-5.5 text-dark backdrop-blur-[2px] transition-colors outline-none placeholder:text-grey-600 hover:border-dark focus-visible:border-dark focus-visible:ring-ring/50 disabled:pointer-events-none disabled:cursor-not-allowed disabled:border-grey-200 disabled:opacity-50 aria-invalid:border-destructive aria-invalid:text-destructive aria-invalid:ring-destructive/20 flex-1 rounded-none border-0 bg-transparent shadow-none ring-0 focus-visible:ring-0 disabled:bg-transparent aria-invalid:ring-0" data-list-empty="" autocomplete="off" spellcheck="false" autocorrect="off" autocapitalize="none" role="combobox" aria-expanded="false" aria-haspopup="listbox" aria-autocomplete="list" id="base-ui-_r_1_" placeholder="Chercher un participant" type="text" value

In [22]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

url = "https://protiming.fr/Results/running/7984-Triathlon-Annette-K/654"
html = requests.get(url, verify=False).text

soup = BeautifulSoup(html, "lxml")

table = soup.find("table")
rows = table.find_all("tr")

data = []

#for row in rows[1:]:  # skip header
for row in rows:  # skip header
    cols = [c.get_text(strip=True) for c in row.find_all("td")]
    if cols:
        data.append(cols)

/Users/maclv/Work/dev/chrono/lib/python3.14/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'protiming.fr'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


In [26]:
headers = []

thead = table.find("thead")
header_rows = thead.find_all("tr")

for tr in header_rows:
    row_headers = []
    for th in tr.find_all("th",class_=lambda c: c != "hidden-xs"):
        text = th.get_text(strip=True)
        colspan = int(th.get("colspan", 1))
        row_headers.extend([text] * colspan)
    headers.append(row_headers)

headers


[['Pos.',
  'Temps',
  'Nom - Prénom',
  'Dossard',
  'Catégorie',
  'Natation',
  'Velo',
  'CaP',
  '']]

In [25]:
tr.find_all("th")

[<th class="hidden-xs" style="">Position</th>,
 <th class="visible-xs" style="">Pos.</th>,
 <th>
 <a href="/Results/running/7984-Triathlon-Annette-K/654/sort:runner_time/direction:asc?url=Results%2Frunning%2F7984-Triathlon-Annette-K%2F654" style="">Temps</a> </th>,
 <th style="color:#D7B56D;font-weight: normal;text-align: left"> <a href="/Results/running/7984-Triathlon-Annette-K/654/sort:last_name/direction:asc?url=Results%2Frunning%2F7984-Triathlon-Annette-K%2F654">Nom - Prénom</a> </th>,
 <th class=""><a href="/Results/running/7984-Triathlon-Annette-K/654/sort:dossard/direction:asc?url=Results%2Frunning%2F7984-Triathlon-Annette-K%2F654">Dossard</a></th>,
 <th class="text-left"><a href="/Results/running/7984-Triathlon-Annette-K/654/sort:categorie/direction:asc?url=Results%2Frunning%2F7984-Triathlon-Annette-K%2F654">Catégorie</a></th>,
 <th>Natation</th>,
 <th>Velo</th>,
 <th>CaP</th>,
 <th>
 </th>]

In [23]:
df = pd.DataFrame(data)
#df.columns = ["Position", "Temps", "Nom", "Dossard", "Categorie", "Natation", "Velo", "CAP", "to_drop"]
df.head()

,0,1,2,3,4,5,6,7,8
0,1,00:29:33,LUDET Cedric,227,HOM,00:05:58,00:16:28,00:07:07,
1,2,00:31:08,BUFFE Julien,75,HOM,00:06:22,00:17:09,00:07:37,
2,3,00:31:52,DE Poorter,163,HOM,00:05:26,00:18:30,00:07:56,
3,4,00:32:00,AERNOUT Louis,23,HOM,00:05:13,00:18:46,00:08:01,
4,5,00:32:41,PLACES Marc,234,HOM,00:06:15,00:17:44,00:08:42,


In [21]:
d = [['1', '00:05:13', 'AERNOUT Louis', '', '23', 'S1M (1 / 28)', '4.6', ''], ['2', '00:05:23', 'RABESANDRATANA Eric', '', '232', 'M3M (1 / 11)', '4.4', ''], ['3', '00:05:26', 'DE POORTER Axel', 'LE MANS TRIATHLON', '163', 'S3M (1 / 53)', '4.4', ''], ['4', '00:05:27', 'OUENNICHE Rim', '', '113', 'S2F (1 / 68)', '4.4', ''], ['5', '00:05:42', 'RIGAUDEAU Thibaut', '', '237', 'S4M (1 / 30)', '4.2', ''], ['6', '00:05:47', 'CAUCHY Olivier', 'THALES CRISTAL TRIATHLON', '17', 'S3M (2 / 53)', '4.1', ''], ['7', '00:05:58', 'LUDET Cedric', '', '227', 'S2M (2 / 68)', '4.0', ''], ['8', '00:06:08', 'CRIADO Guillaume', 'AVIRON BAYONNAIS TRIATHLON', '201', 'S3M (3 / 53)', '3.9', ''], ['9', '00:06:09', 'FREON Axel', 'ANNETTE K', '56', 'S3M (4 / 53)', '3.9', ''], ['10', '00:06:10', 'VEYNE Julien', '', '209', 'S4M (2 / 30)', '3.8', ''], ['11', '00:06:15', 'PLACES Marc', '', '234', 'S2M (3 / 68)', '3.8', ''], ['12', '00:06:15', 'MALARY Antoine', '', '20', 'S2M (4 / 68)', '3.8', ''], ['13', '00:06:22', 'BUFFE Julien', '', '75', 'S4M (3 / 30)', '3.7', ''], ['14', '00:06:23', 'BONNET Mathieu', 'PARIS UNIVERSITE CLUB', '151', 'M2M (1 / 6)', '3.7', ''], ['15', '00:06:24', 'MAMOU Thomas', 'PREFONTAINE', '215', 'S4M (4 / 30)', '3.7', ''], ['16', '00:06:25', 'DELAUX Marie', '', '213', 'S3F (5 / 53)', '3.7', ''], ['17', '00:06:25', 'LONGAYGUE Hermance', '', '134', 'S2F (5 / 68)', '3.7', ''], ['18', '00:06:27', 'TRIHAN Malo', 'THALES', '19', 'S2M (6 / 68)', '3.7', ''], ['19', '00:06:32', 'SAUVIAT Marc-antoine', 'LEVALLOIS TRIATHLON', '170', 'S3M (6 / 53)', '3.6', ''], ['20', '00:06:36', 'BONNET Jonathan', 'U S  CRETEIL TRIATHLON', '214', 'S2M (7 / 68)', '3.6', ''], ['21', '00:06:37', 'JANUS Stephanie', '', '190', 'S4F (5 / 30)', '3.6', ''], ['22', '00:06:39', 'VON ROCHOW Amelie', '', '80', 'S4F (6 / 30)', '3.6', ''], ['23', '00:06:46', 'TEISSIER Jean baptiste', 'MEUDON TRIATHLON', '219', 'M2M (2 / 6)', '3.5', ''], ['24', '00:06:51', 'FOURDINIER Edouard', 'ISSY TRIATHLON', '211', 'S3M (7 / 53)', '3.5', ''], ['25', '00:06:52', 'MAHAJOUB Gregory', 'MEUDON TRIATHLON', '59', 'M3M (2 / 11)', '3.4', ''], ['26', '00:06:54', 'DURAND Eric', 'ANNETTEK', '217', 'M3M (3 / 11)', '3.4', ''], ['27', '00:06:55', 'BERRADA Salma', '', '197', 'S3F (8 / 53)', '3.4', ''], ['28', '00:06:56', 'MILLET Arthur', '', '90', 'S3M (9 / 53)', '3.4', ''], ['29', '00:06:58', 'JAMES Molly', '', '119', 'S2F (8 / 68)', '3.4', ''], ['30', '00:07:00', 'CARDENAS MENDOZA Stephanie', 'USMA SAINT OUEN', '122', 'S3F (10 / 53)', '3.4', ''], ['31', '00:07:00', 'SPIGOLON Chloe', '', '83', 'S2F (9 / 68)', '3.4', ''], ['32', '00:07:02', 'RODIER Adrien', '', '186', 'S2M (10 / 68)', '3.4', ''], ['33', '00:07:06', 'ZERWETZ Aude', '', '111', 'S3F (11 / 53)', '3.3', ''], ['34', '00:07:06', 'PONSARD Francois', 'KINE SPORT SANTE PARIS 14', '123', 'S2M (11 / 68)', '3.3', ''], ['35', '00:07:07', 'KHATTAR Marine', '', '27', 'S1F (2 / 28)', '3.3', ''], ['36', '00:07:08', 'VOGLEY Mathieu', '', '185', 'M1M (1 / 6)', '3.3', ''], ['37', '00:07:09', 'DORVAL Albert', '', '120', 'S4M (7 / 30)', '3.3', ''], ['38', '00:07:13', 'GRAVIER Alex', 'ISSY TRIATHLON', '42', 'S2M (12 / 68)', '3.3', ''], ['39', '00:07:19', 'RENAULT Julien', 'GC37', '129', 'S3M (12 / 53)', '3.2', ''], ['40', '00:07:20', 'MULLER Bernard', 'MEUDON TRIATHLON', '58', 'M4M (1 / 2)', '3.2', ''], ['41', '00:07:21', 'RULLIER Louis', '', '150', 'S4M (8 / 30)', '3.2', ''], ['42', '00:07:22', 'STOURM Maugan', '', '132', 'S2M (13 / 68)', '3.2', ''], ['43', '00:07:25', 'VENTRE Sylvain', '', '228', 'M1M (2 / 6)', '3.2', ''], ['44', '00:07:27', 'CHASSIGNEUX Alexia', 'THALES CRISTAL', '16', 'S3F (13 / 53)', '3.2', ''], ['45', '00:07:27', 'LE GUEN Mathieu', 'THALES CRISTAL TRIATHLON', '43', 'S2M (14 / 68)', '3.2', ''], ['46', '00:07:30', 'ROUX Lucille', '', '3', 'S3F (14 / 53)', '3.2', ''], ['47', '00:07:30', 'ADAMOWSKI Kelly', '', '114', 'S3F (15 / 53)', '3.2', ''], ['48', '00:07:31', 'PROUILHAC Julien', '', '112', 'S4M (9 / 30)', '3.1', ''], ['49', '00:07:34', 'HENRI Tiphaine', 'PARIS UNIVERSITE CLUB', '152', 'M1F (3 / 6)', '3.1', ''], ['50', '00:07:37', 'STOURM Meliau', 'LES GARS-LACTIQUES', '144', 'S1M (3 / 28)', '3.1', ''], ['51', '00:07:39', 'HARET Alexandre', '', '69', 'S3M (16 / 53)', '3.1', ''], ['52', '00:07:39', 'MA Brandon', '', '220', 'S2M (15 / 68)', '3.1', ''], ['53', '00:07:41', 'ALANDETE Juan', '', '15', 'S4M (10 / 30)', '3.1', ''], ['54', '00:07:43', 'GERVASIO Alexandre', 'ANNETTE K', '66', 'S2M (16 / 68)', '3.1', ''], ['55', '00:07:44', 'HENRY Charlotte', 'PARIS AQUATIQUE', '175', 'S4F (11 / 30)', '3.1', ''], ['56', '00:07:46', 'HOUMEAU Alexandre', '', '98', 'S3M (17 / 53)', '3.0', ''], ['57', '00:07:47', 'POULAIN Pierre', 'U S  CRETEIL TRIATHLON', '212', 'S3M (18 / 53)', '3.0', ''], ['58', '00:07:48', 'GUTTON Alexandre', '', '78', 'S3M (19 / 53)', '3.0', ''], ['59', '00:07:50', 'RENAULT Clement', '', '131', 'S2M (17 / 68)', '3.0', ''], ['60', '00:07:50', 'VAN DRIESSCHE Jade', '', '67', 'S2F (18 / 68)', '3.0', ''], ['61', '00:07:52', 'RIBEIRO Antonin', '', '74', 'S2M (19 / 68)', '3.0', ''], ['62', '00:07:53', 'JENKINS Maureen', '', '238', 'S2F (20 / 68)', '3.0', ''], ['63', '00:07:55', 'GEZEGOU Romain', '', '167', 'S3M (20 / 53)', '3.0', ''], ['64', '00:07:57', 'LEFRANCOIS Chiara', '', '126', 'S1F (4 / 28)', '3.0', ''], ['65', '00:07:57', 'MRAD Sofyan', '', '76', 'S3M (21 / 53)', '3.0', ''], ['66', '00:08:02', 'RIVIERE Cyril', 'LES EMPLAQUES', '140', 'S4M (12 / 30)', '2.9', ''], ['67', '00:08:03', 'ROLLIN Margaux', '', '40', 'S1F (5 / 28)', '2.9', ''], ['68', '00:08:08', 'LEFEBVRE Alexandre', '', '103', 'S3M (22 / 53)', '2.9', ''], ['69', '00:08:09', 'CHAMPION Thibaut', '', '147', 'S2M (21 / 68)', '2.9', ''], ['70', '00:08:11', 'BERNARD Julien', 'DASSAULT SPORTS', '101', 'S4M (13 / 30)', '2.9', ''], ['71', '00:08:11', 'LEFEBVRE Mathilde', 'TRIATHLON CLUB SOISSONS', '203', 'S3F (23 / 53)', '2.9', ''], ['72', '00:08:12', 'LE GARREC Erwan', '', '100', 'S3M (24 / 53)', '2.9', ''], ['73', '00:08:12', 'GARNIER Anouk', '', '226', 'S2F (22 / 68)', '2.9', ''], ['74', '00:08:13', 'DUARTE Mathieu', 'ATHLETIC CLUB BOULOGNE BILLANC..', '63', 'S3M (25 / 53)', '2.9', ''], ['75', '00:08:13', 'LIF Yanis', 'THALES CRISTAL TRIATHLON', '22', 'S3M (26 / 53)', '2.9', ''], ['76', '00:08:14', 'OGER Marie', '', '191', 'M1F (4 / 6)', '2.9', ''], ['77', '00:08:16', 'LEVY Noemie', '', '164', 'S2F (23 / 68)', '2.9', ''], ['78', '00:08:16', 'PASQUIER Tom', '', '70', 'S1M (6 / 28)', '2.9', ''], ['79', '00:08:17', 'MERRAN Gary', '', '30', 'S2M (24 / 68)', '2.8', ''], ['80', '00:08:19', 'DESLANDES Laetitia', '', '34', 'S3F (27 / 53)', '2.8', ''], ['81', '00:08:19', 'MAINGUY Violaine', 'ATHLETIC CLUB BOULOGNE BILLANC..', '26', 'M2F (3 / 6)', '2.8', ''], ['82', '00:08:20', 'CHRISTIEN Gaetan', 'VERSAILLES TRIATHLON', '51', 'S3M (28 / 53)', '2.8', ''], ['83', '00:08:24', 'GABELLE Johann', '', '107', 'S4M (14 / 30)', '2.8', ''], ['84', '00:08:24', 'BUFFETEAU Luigi', '', '49', 'S2M (25 / 68)', '2.8', ''], ['85', '00:08:24', 'BESSON Guillaume', 'FRONT RUNNERS PARIS', '202', 'S4M (15 / 30)', '2.8', ''], ['86', '00:08:27', 'DI CARLO Melissa', '', '199', 'S3F (29 / 53)', '2.8', ''], ['87', '00:08:30', 'BIARD Luna', '', '84', 'S2F (26 / 68)', '2.8', ''], ['88', '00:08:30', 'CARRE Lenaig', '', '222', 'S2F (27 / 68)', '2.8', ''], ['89', '00:08:31', 'CHETANEAU Angelique', '', '235', 'S4F (16 / 30)', '2.8', ''], ['90', '00:08:32', 'ZEIDLER-CLUZEL Timothe', '', '125', 'S1M (7 / 28)', '2.8', ''], ['91', '00:08:34', 'MAZUR Hanna', '', '161', 'S4F (17 / 30)', '2.8', ''], ['92', '00:08:34', 'POL Louka', '', '18', 'S2M (28 / 68)', '2.8', ''], ['93', '00:08:35', 'MARGARIT Anais', 'TRIATHLON CLUB SOISSONS', '204', 'S4F (18 / 30)', '2.8', ''], ['94', '00:08:37', 'PELTIER Fanny', '', '233', 'S2F (29 / 68)', '2.7', ''], ['95', '00:08:39', 'ROTH Dominique', '', '81', 'S3M (30 / 53)', '2.7', ''], ['96', '00:08:39', 'SEBAN Julie', 'THE GOOD TRAINING', '165', 'S4F (19 / 30)', '2.7', ''], ['97', '00:08:40', 'PERRIER Gabrielle', '', '174', 'S1F (8 / 28)', '2.7', ''], ['98', '00:08:45', 'CORON Camille', '', '138', 'S2F (30 / 68)', '2.7', ''], ['99', '00:08:47', 'MAHAJOUB Cian', '', '65', 'S1M (9 / 28)', '2.7', ''], ['100', '00:08:48', 'DAVID Hugo', 'ASTC CRISTAL', '31', 'S2M (31 / 68)', '2.7', ''], ['101', '00:08:49', 'DELMELLE Laetitia', '', '61', 'M2F (4 / 6)', '2.7', ''], ['102', '00:08:50', 'BIGOU Elise', '', '121', 'S3F (31 / 53)', '2.7', ''], ['103', '00:08:50', 'CALIO Jean', '', '1', 'M3M (4 / 11)', '2.7', ''], ['104', '00:08:54', 'BERTOLO Line', '', '137', 'S2F (32 / 68)', '2.6', ''], ['105', '00:08:55', 'KAMRATZKI Juliette', '', '85', 'S2F (33 / 68)', '2.6', ''], ['106', '00:08:57', 'BOUCHER Valentin', '', '21', 'S2M (34 / 68)', '2.6', ''], ['107', '00:08:57', 'HUGUET Laurie', 'THALES CRISTAL', '32', 'S3F (32 / 53)', '2.6', ''], ['108', '00:08:57', 'ROSAY Alex', '', '135', 'S2M (35 / 68)', '2.6', ''], ['109', '00:08:58', 'BLANCHET Marine', '', '47', 'S2F (36 / 68)', '2.6', ''], ['110', '00:08:59', 'GALLMANN Guillaume', '', '7', 'S3M (33 / 53)', '2.6', ''], ['111', '00:08:59', 'SESEKE Paul', '', '156', 'S2M (37 / 68)', '2.6', ''], ['112', '00:09:02', 'HALL Stephen', '', '68', 'S3M (34 / 53)', '2.6', ''], ['113', '00:09:02', 'FONTAINE Pierre', '', '127', 'S2M (38 / 68)', '2.6', ''], ['114', '00:09:05', 'GUITTOT Marine', '', '110', 'S2F (39 / 68)', '2.6', ''], ['115', '00:09:05', 'PAILLOUX Amaury', '', '94', 'S1M (10 / 28)', '2.6', ''], ['116', '00:09:09', 'DUBOIS Coralie', 'STADE FRANCAIS', '218', 'S3F (35 / 53)', '2.6', ''], ['117', '00:09:09', 'ROSAY Chloe', '', '109', 'S3F (36 / 53)', '2.6', ''], ['118', '00:09:12', 'NOTAIS Esteban', '', '50', 'S3M (37 / 53)', '2.6', ''], ['119', '00:09:12', 'FAUSSEMAGNE Magali', '', '188', 'S4F (20 / 30)', '2.6', ''], ['120', '00:09:15', 'BERNARD Benoit-joseph', '2B-INVESTISSEMENTS', '162', 'S3M (38 / 53)', '2.5', ''], ['121', '00:09:15', 'DAVID Valentin', '', '86', 'S3M (39 / 53)', '2.5', ''], ['122', '00:09:16', 'CZUBAK Sebastien', 'VERSAILLES TRIATHLON', '95', 'M2M (5 / 6)', '2.5', ''], ['123', '00:09:17', 'GOLDSCHMIDT Lea', '', '39', 'S2F (40 / 68)', '2.5', ''], ['124', '00:09:17', 'PREVOST Tiphaine', '', '24', 'S1F (11 / 28)', '2.5', ''], ['125', '00:09:18', 'FOEILLET Victoire', '', '176', 'S2F (41 / 68)', '2.5', ''], ['126', '00:09:19', 'BRICE Hugo', '', '9', 'S2M (42 / 68)', '2.5', ''], ['127', '00:09:21', 'SESEKE Carla', '', '154', 'S1F (12 / 28)', '2.5', ''], ['128', '00:09:23', 'CLERC Laura', '', '38', 'S2F (43 / 68)', '2.5', ''], ['129', '00:09:26', 'CANIVET Justine', '', '36', 'S4F (21 / 30)', '2.5', ''], ['130', '00:09:26', 'PETITJEAN Loic', '', '160', 'S1M (13 / 28)', '2.5', ''], ['131', '00:09:27', 'VAN DRIESSCHE Brice', '', '108', 'S2M (44 / 68)', '2.5', ''], ['132', '00:09:30', 'PYPAERT Jean-matthieu', '', '153', 'S2M (45 / 68)', '2.5', ''], ['133', '00:09:31', 'MOLINA Cleo', '', '115', 'S2F (46 / 68)', '2.5', ''], ['134', '00:09:32', 'BOUCHAMA Imene', '', '195', 'S4F (22 / 30)', '2.5', ''], ['135', '00:09:33', 'TUFFIN Alexandre', '', '142', 'S1M (14 / 28)', '2.5', ''], ['136', '00:09:33', 'DUVAL Jordan', '', '240', 'S4M (23 / 30)', '2.5', ''], ['137', '00:09:33', 'FREY Yoann', '', '171', 'S2M (47 / 68)', '2.5', ''], ['138', '00:09:37', 'BEN OTHMAN Sarah', '', '184', 'M1F (5 / 6)', '2.4', ''], ['139', '00:09:38', 'LEONARD Florence', '', '79', 'S3F (40 / 53)', '2.4', ''], ['140', '00:09:39', 'RADIGON Juliette', '', '105', 'S1F (15 / 28)', '2.4', ''], ['141', '00:09:39', 'BERTIN Corentin', '', '29', 'S1M (16 / 28)', '2.4', ''], ['142', '00:09:41', 'MORVAN Tiphaine', '', '133', 'S2F (48 / 68)', '2.4', ''], ['143', '00:09:45', 'O DONNELL Jaime', '', '182', 'S3F (41 / 53)', '2.4', ''], ['144', '00:09:48', 'LLONIN Amandine', '', '2', 'S2F (49 / 68)', '2.4', ''], ['145', '00:09:49', 'HOUY Maxane', '', '48', 'S2F (50 / 68)', '2.4', ''], ['146', '00:09:50', 'CURIN Theo', '', '229', 'S2M (51 / 68)', '2.4', ''], ['147', '00:09:55', 'BLANGERO Lucas', '', '88', 'S3M (42 / 53)', '2.4', ''], ['148', '00:09:56', 'VIDAL Solene', '', '93', 'S1F (17 / 28)', '2.4', ''], ['149', '00:09:58', 'KATUGAMPALA Udara', '', '166', 'S2M (52 / 68)', '2.4', ''], ['150', '00:10:00', 'GUILLOT Anne frederique', '', '148', 'M3F (5 / 11)', '2.4', ''], ['151', '00:10:05', 'ZAETTA Paul', '', '206', 'S3M (43 / 53)', '2.3', ''], ['152', '00:10:06', 'LAUSSUCQ Mathilde', '', '221', 'S2F (53 / 68)', '2.3', ''], ['153', '00:10:07', 'DROUIN Hugo', '', '35', 'S2M (54 / 68)', '2.3', ''], ['154', '00:10:09', 'MICHARD RIVIERE Eva', 'LEVALLOIS TRIATHLON', '145', 'S3F (44 / 53)', '2.3', ''], ['155', '00:10:14', 'PELSEZ-MAHAJOUB Sarah', 'MEUDON TRIATHLON', '60', 'M3F (6 / 11)', '2.3', ''], ['156', '00:10:15', 'SALEK Samira', '', '179', 'S1F (18 / 28)', '2.3', ''], ['157', '00:10:17', 'DUMONT Chrystele', 'RMA', '64', 'M3F (7 / 11)', '2.3', ''], ['158', '00:10:17', 'SAILLARD Hildegarde', '', '169', 'S2F (55 / 68)', '2.3', ''], ['159', '00:10:18', 'EMMANUEL Dagou', '', '181', 'S3M (45 / 53)', '2.3', ''], ['160', '00:10:20', 'ROCHMANN Jean-baptiste', 'JB CORP', '89', 'S3M (46 / 53)', '2.3', ''], ['161', '00:10:21', 'ESTEVE Manon', '', '28', 'S2F (56 / 68)', '2.3', ''], ['162', '00:10:28', 'MAHIER Julie', '', '33', 'S2F (57 / 68)', '2.2', ''], ['163', '00:10:28', 'MOUSSARD Laura', '', '124', 'S1F (19 / 28)', '2.2', ''], ['164', '00:10:30', 'POHU Jeremy', '', '207', 'S3M (47 / 53)', '2.2', ''], ['165', '00:10:31', 'BERNAY Marie-alix', 'UNEXPECTED', '12', 'S4F (24 / 30)', '2.2', ''], ['166', '00:10:33', 'LAGANE Camille', '', '91', 'S4F (25 / 30)', '2.2', ''], ['167', '00:10:36', 'ISAMBERT Leonore', '', '37', 'S1F (20 / 28)', '2.2', ''], ['168', '00:10:37', 'GICQUEL Emma', '', '102', 'S2F (58 / 68)', '2.2', ''], ['169', '00:10:40', 'RESTELLINI Jeremie', '', '177', 'S3M (48 / 53)', '2.2', ''], ['170', '00:10:44', 'BAVELIER Clemence', '', '82', 'S3F (49 / 53)', '2.2', ''], ['171', '00:10:48', 'BERHAULT Mevena', '', '41', 'S1F (21 / 28)', '2.2', ''], ['172', '00:10:55', 'ROUSSILLON Marius', '', '73', 'S3M (50 / 53)', '2.2', ''], ['173', '00:10:59', 'LE BRIGAND Pierre', 'PARIS UNIVERSITE CLUB', '205', 'S1M (22 / 28)', '2.1', ''], ['174', '00:11:04', 'BOUSQUET Romane', '', '136', 'S1F (23 / 28)', '2.1', ''], ['175', '00:11:10', 'FROMENT Laetitia', '', '11', 'M2F (6 / 6)', '2.1', ''], ['176', '00:11:21', 'NAVELLOU Arnaud', '', '53', 'S1M (24 / 28)', '2.1', ''], ['177', '00:11:22', 'DIETRICH Maxime', 'OTEA PRODUCTION', '96', 'S4M (26 / 30)', '2.1', ''], ['178', '00:11:25', 'DUPUY Emmanuelle', '', '149', 'M3F (8 / 11)', '2.1', ''], ['179', '00:11:29', 'NICOLAS Sylvain', '', '230', 'S4M (27 / 30)', '2.0', ''], ['180', '00:11:32', 'RAVELEAU Angele', '', '72', 'S2F (59 / 68)', '2.0', ''], ['181', '00:11:33', 'CAILLON Tristan', '', '4', 'S1M (25 / 28)', '2.0', ''], ['182', '00:11:34', 'RAMBLAS Elsa', '', '118', 'S2F (60 / 68)', '2.0', ''], ['183', '00:11:37', 'COLLET Josephine', '', '10', 'S1F (26 / 28)', '2.0', ''], ['184', '00:11:51', 'CONAN Emma', '', '128', 'S2F (61 / 68)', '2.0', ''], ['185', '00:11:53', 'ROPARS Lisa-marie', 'LES GARS-LACTIQUES', '143', 'S1F (27 / 28)', '2.0', ''], ['186', '00:11:58', 'SELVI Claire', '', '159', 'S2F (62 / 68)', '2.0', ''], ['187', '00:12:07', 'VIDAL Clemence', '', '92', 'S2F (63 / 68)', '1.9', ''], ['188', '00:12:08', 'NIOPEL Romane', '', '141', 'S2F (64 / 68)', '1.9', ''], ['189', '00:12:15', 'SAMII Eva', '', '62', 'M3F (9 / 11)', '1.9', ''], ['190', '00:12:27', 'PAYCHA Chloee', '', '87', 'S1F (28 / 28)', '1.9', ''], ['191', '00:12:33', 'JAULT-SESEKE Fabienne', '', '155', 'M4F (2 / 2)', '1.9', ''], ['192', '00:13:08', 'GROJEAN Muriel', '', '192', 'S4F (28 / 30)', '1.8', ''], ['193', '00:13:21', 'PREVOST Solene', '', '25', 'S2F (65 / 68)', '1.7', ''], ['194', '00:13:21', 'BLAIN Charlene', '', '13', 'S2F (66 / 68)', '1.7', ''], ['195', '00:13:23', 'THIEBLEMONT Morgane', '', '139', 'S4F (29 / 30)', '1.7', ''], ['196', '00:13:30', 'CAUDRON Emmanuelle', 'ANNETTE K', '55', 'M3F (10 / 11)', '1.7', ''], ['197', '00:13:32', 'CONRAUX Pierre', 'IRA', '193', 'S3M (51 / 53)', '1.7', ''], ['198', '00:14:29', 'DUPARD Marina', '', '198', 'S3F (52 / 53)', '1.6', ''], ['199', '00:14:59', 'PERRETTE Mathilde', '', '168', 'S2F (67 / 68)', '1.6', ''], ['200', '00:15:26', 'VOINOT Sebastien', '', '189', 'M3M (11 / 11)', '1.5', ''], ['201', '00:17:44', 'CORSON Lenaig', '', '236', 'S4F (30 / 30)', '1.3', ''], ['202', '00:17:52', 'FOURNIER Emma', '', '216', 'S2F (68 / 68)', '1.3', ''], ['203', '00:18:50', 'MEITE Bakary', '', '231', 'M1M (6 / 6)', '1.2', ''], ['204', '00:19:46', 'TAPIA Nicolas', '', '200', 'S3M (53 / 53)', '1.2', '']]
pd.DataFrame(d)

,0,1,2,3,4,5,6,7
0,1,00:05:13,AERNOUT Louis,,23,S1M (1 / 28),4.6,
1,2,00:05:23,RABESANDRATANA Eric,,232,M3M (1 / 11),4.4,
2,3,00:05:26,DE POORTER Axel,LE MANS TRIATHLON,163,S3M (1 / 53),4.4,
3,4,00:05:27,OUENNICHE Rim,,113,S2F (1 / 68),4.4,
4,5,00:05:42,RIGAUDEAU Thibaut,,237,S4M (1 / 30),4.2,
5,6,00:05:47,CAUCHY Olivier,THALES CRISTAL TRIATHLON,17,S3M (2 / 53),4.1,
6,7,00:05:58,LUDET Cedric,,227,S2M (2 / 68),4.0,
7,8,00:06:08,CRIADO Guillaume,AVIRON BAYONNAIS TRIATHLON,201,S3M (3 / 53),3.9,
8,9,00:06:09,FREON Axel,ANNETTE K,56,S3M (4 / 53),3.9,
9,10,00:06:10,VEYNE Julien,,209,S4M (2 / 30),3.8,


# CHRONOWEB

In [20]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

url = "https://chronoweb.com/resultats.php?annee=2008"
resp = requests.get(url)
resp.raise_for_status()

soup = BeautifulSoup(resp.text, "html.parser")
base_url = "https://chronoweb.com"

xls_links = []
for a in soup.find_all("a", href=True):
    #print(a["href"])
    href = a["href"].lower()
    if href.endswith('.xls') or href.endswith('.xlsx'):
        full_url = urljoin(base_url, a["href"])
        xls_links.append({
            "text": a.get_text(strip=True),
            "url": full_url
        })

# en DataFrame
import pandas as pd
df = pd.DataFrame(xls_links)
#print(df)

In [21]:
df

,text,url
0,XLS,https://chronoweb.com/files/xls/Resultats_Triathlon_dOlron_2025.xlsx
1,XLS,https://chronoweb.com/files/xls/Rsultats_ALEFPA_TRAIL_2025.xlsx
2,XLS,https://chronoweb.com/files/xls/Resultats_Extrememan_2025.xlsx
3,XLS,https://chronoweb.com/files/xls/Resultats_Triathlon_de_St_Pardoux_2025.xlsx
4,XLS,https://chronoweb.com/files/xls/Resultats_Triathlon_de_lOmois_2025.xlsx
5,XLS,https://chronoweb.com/files/xls/Resultats_Triathlon_du_Levezou_2025.xlsx
6,XLS,https://chronoweb.com/files/xls/Resultats_Triathlon_de_Vouglans_2025.xlsx
7,XLS,https://chronoweb.com/files/xls/Resultats_Triathlon_de_Nevers_2025.xlsx
8,XLS,https://chronoweb.com/files/xls/Resultats_Swimrun_Levezou_2025.xlsx
9,XLS,https://chronoweb.com/files/xls/Resultats_St_Jouvent_2025.xlsx


In [71]:
def extract_race_info(url):
    path = urlparse(url).path
    parts = path.strip("/").split("/")
    slug = os.path.splitext(parts[2])
    infos = slug[0].split("_")[1:]
    name = " ".join(infos[:-1])
    year = infos[-1]
    return name, year

extract_race_name("https://chronoweb.com/files/xls/Rsultats_Nevers_2020.xls")

('Nevers', '2020')

In [123]:
# récupération du fichier de résultat

In [124]:
response = requests.get("https://chronoweb.com/files/xls/Resultats_Triathlon_dOlron_2025.xlsx",verify=False)

# Save the file
with open("../../Downloads/file.xlsx", "wb") as file:
    file.write(response.content)

/Users/maclv/Work/dev/chrono/lib/python3.12/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'chronoweb.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


In [125]:
file_name = "../../Downloads/file.xlsx"
xl_file = pd.ExcelFile(file_name)

ImportError: Missing optional dependency 'openpyxl'.  Use pip or conda to install openpyxl.

In [53]:
xl_file.sheet_names

['M', 'M Relais', 'XS']

In [54]:
dfs = pd.read_excel(file_name, sheet_name='M')

In [55]:
dfs

,Unnamed: 0,Unnamed: 1,Unnamed: 2,Unnamed: 3,Arrivée,Unnamed: 5,Unnamed: 6,Natation,Unnamed: 8,Unnamed: 9,...,Vélo,Unnamed: 12,Unnamed: 13,Unnamed: 14,Unnamed: 15,Unnamed: 16,CàP,Unnamed: 18,Unnamed: 19,Unnamed: 20
0,Clt,Nom,Club,Dos,Temps,Cat,NaN,Temps,Moy,Clt,...,Temps,Moy,Clt,FinV,GàV,TR2,Temps,Moy,Clt,GàP
1,1,Hugo POULMARCH,LES GIRONDINS DE BORDEAUX TRIATHLON,246,02:03:41,1/MSE,NaN,00:23:10,1:32,3,...,01:03:09,37.3,2,2,1,00:01:28,00:34:39,17.3,1,1
2,2,Antoine BOULEGUE,AST TRIATHLON LA TESTE,190,02:07:36,2/MSE,NaN,00:24:45,1:39,10,...,01:04:55,36.3,6,3,7,00:01:25,00:35:27,16.9,2,1
3,3,Nael DOS SANTOS,MONTPELLIER TRIATHLON,153,02:10:20,3/MSE,NaN,00:20:10,1:20,1,...,01:02:56,37.4,1,1,0,00:01:27,00:44:42,13.4,39,-2
4,4,Jean christophe LONGEARD,AST TRIATHLON LA TESTE,191,02:12:15,1/MVE,NaN,00:24:28,1:37,9,...,01:06:56,35.2,10,7,2,00:01:31,00:38:00,15.7,4,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
250,DNS,Thomas BRUTIN,NaN,141,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
251,DNS,Alexis COUDERT,NaN,161,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
252,DNS,Pierre emmanuel GALLEGO,CLERMONT TRIATHLON,182,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
253,DNS,Alexandre RIVY,US PALAISEAU TRIATHLON,194,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [29]:
pd.read_excel("../dev/sources/chronoweb/tmp_file.xlsx")

,Unnamed: 0,Unnamed: 1,Unnamed: 2,Unnamed: 3,Arrivée,Unnamed: 5,Unnamed: 6,Natation,Unnamed: 8,Unnamed: 9,Unnamed: 10,Vélo,Unnamed: 12,Unnamed: 13,Unnamed: 14,Unnamed: 15,Unnamed: 16,CàP,Unnamed: 18,Unnamed: 19,Unnamed: 20
0,Clt,Nom,Club,Dos,Temps,Cat,NaN,Temps,Moy,Clt,TR1,Temps,Moy,Clt,FinV,GàV,TR2,Temps,Moy,Clt,GàP
1,1,Paul DELAMARE,ROUEN TRIATHLON,52,01:59:32,1/MJU,NaN,00:18:56,1:15,1,00:05:37,00:57:27,41.7,1,1,0,00:02:38,00:34:52,17.2,2,0
2,2,Nicolas ONILLON,SHARKS TRI 79,199,02:10:08,1/MVE,NaN,00:22:09,1:28,2,00:06:21,01:00:48,39.4,4,2,0,00:02:51,00:37:57,15.8,6,0
3,3,Francois BODIC,SAINT AVERTIN SPORTS TRIATHLON 37,45,02:12:00,2/MVE,NaN,00:26:54,1:47,14,00:06:26,00:58:33,40.9,2,3,11,00:02:32,00:37:33,15.9,4,0
4,4,Arthur BAUDUIN,NaN,15,02:15:52,1/MSE,NaN,00:28:39,1:54,25,00:06:52,01:00:41,39.5,3,6,19,00:02:32,00:37:07,16.1,3,2
5,5,Matteo DOUSSET,LA ROCHELLE TRIATHLON,260,02:18:43,2/MSE,NaN,00:22:19,1:29,3,00:06:30,01:04:17,37.3,12,4,-1,00:03:34,00:42:00,14.2,18,-1
6,6,Hugo NEDELLEC,ROUEN TRIATHLON,53,02:23:26,2/MJU,NaN,00:22:20,1:29,4,00:06:11,01:05:43,36.5,23,5,-1,00:03:02,00:46:08,13.0,61,-1
7,7,Robin GENDROT,TRIATHLON CLUB DE LIEVIN,357,02:23:29,3/MSE,NaN,00:26:07,1:44,8,00:08:29,01:01:46,38.8,6,7,1,00:03:41,00:43:23,13.8,29,0
8,8,Jonathan MOREAU,NaN,3,02:24:03,3/MVE,NaN,00:26:11,1:44,9,00:07:43,01:02:31,38.3,7,8,1,00:03:39,00:43:57,13.6,36,0
9,9,Erwan QUEMENEUR,NaN,148,02:24:22,4/MSE,NaN,00:32:53,2:11,105,00:07:50,01:02:39,38.3,8,14,91,00:03:03,00:37:54,15.8,5,5


# FFTRI T2 Area

In [6]:
# récupération de toutes les épreuves

In [26]:
def extract_links(driver):
    """Récupère les épreuves via les liens class='edition btn'."""
    links = []
    elements = driver.find_elements(By.CSS_SELECTOR, "a.edition.btn")
    for el in elements:
        link = urljoin(BASE_URL, el.get_attribute("href"))
        links.append(link)
    return links
    
#current_links = extract_links()
#print(current_links)

In [28]:
BASE_URL = "https://fftri.t2area.com"
START_URL = "https://fftri.t2area.com/resultats.html"

# --- Setup du driver ---
options = webdriver.ChromeOptions()
options.add_argument("--headless")  # retire cette ligne pour voir le navigateur
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

driver.get(START_URL)

SCROLL_PAUSE = 1.5  # temps d'attente après chaque scroll

# --- Infinite scroll ---
last_count = 0
same_count_times = 0   # si ça ne bouge plus après X tours → fin

while True:
    # Scroll jusqu'en bas
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(SCROLL_PAUSE)

    # Récupère les épreuves après scroll
    links = extract_links(driver)
    new_count = len(links)

    print(f"Épreuves chargées : {new_count}")

    if new_count == last_count:
        same_count_times += 1
    else:
        same_count_times = 0

    # Si aucun nouvel élément sur 3 cycles → fin
    if same_count_times >= 3:
        print("Le scroll n'ajoute plus d'épreuves → fin du scraping.")
        break

    last_count = new_count

driver.quit()

# Suppression des doublons éventuels
links = list(set(links))

#print("\n=== ÉPREUVES RÉCOLTÉES ===")
#for link in links:
#    print(f"{link}")

print(f"\nTotal : {len(links)} épreuves")


Épreuves chargées : 103
Épreuves chargées : 291
Épreuves chargées : 429
Épreuves chargées : 535
Épreuves chargées : 649
Épreuves chargées : 750
Épreuves chargées : 845
Épreuves chargées : 900
Épreuves chargées : 937
Épreuves chargées : 970
Épreuves chargées : 1123
Épreuves chargées : 1286
Épreuves chargées : 1475
Épreuves chargées : 1602
Épreuves chargées : 1801
Épreuves chargées : 2022
Épreuves chargées : 2382
Épreuves chargées : 2615
Épreuves chargées : 2615
Épreuves chargées : 2615
Épreuves chargées : 2615
Le scroll n'ajoute plus d'épreuves → fin du scraping.

Total : 2615 épreuves


In [31]:
links[:3]

['https://fftri.t2area.com/calendrier/triathlon-de-nouatre/triathlon-m/2019.html',
 'https://fftri.t2area.com/calendrier/triathlon-de-saint-remy-sur-durolle/triathlon-l/2005.html',
 'https://fftri.t2area.com/calendrier/ironman-de-vichy/triathlon-xxl/2011.html']

In [7]:
def extract_date_from_text(text):
    pattern = r"(\d{1,2}\s+[A-Za-zéèêûôîçÉÈÊÛÔÎÇ]+\s+\d{4})"
    m = re.search(pattern, text)
    if m:
        date_str = m.group(1)
        day, month_fr, year = date_str.lower().split()
        return date(int(year), MONTHS_FR[month_fr], int(day))

url = "https://fftri.t2area.com/calendrier/triathlon-de-caen/triathlon-xs/2022.html"

resp = requests.get(url)
resp.raise_for_status()
soup = BeautifulSoup(resp.text, "html.parser")
article = soup.find("article", class_="edition-description-content")
print(article)
if article:
    text = article.get_text(separator="\n", strip=True)
    name_epreuve = text.split("\n")[1]
    date_epreuve = extract_date_from_text(text.split("\n")[2])

None


In [5]:
# récupération du tableau des résultats de l'épreuve

In [6]:
URL = "https://fftri.t2area.com/calendrier/triathlon-de-baudreix/triathlon-l/2023.html"
URL = "https://fftri.t2area.com/calendrier/triathlon-de-baudreix/triathlon-l/2015.html"
URL = "https://fftri.t2area.com/calendrier/championnat-de-france-fftri/triathlon-l/2011.html"
BASE = "https://fftri.t2area.com"

response = requests.get(URL)
response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")

table = soup.find("table", class_="dataTable")

if not table:
    print("Tableau dataTable introuvable.")
else:
    # En-têtes
    headers = [th.text.strip() for th in table.find("thead").find_all("th")]

    rows = []
    for tr in table.find("tbody").find_all("tr"):
        row = {}
        tds = tr.find_all("td")

        print(tds)

        for i, header in enumerate(headers):
            cell = tds[i]

            # --- Cas spécial : colonne "Détails" ---
            if header.lower().startswith("dét"):
                a = cell.find("a")
                if a and a.get("href"):
                    row[header] = urljoin(BASE, a.get("href"))
                else:
                    row[header] = None
            else:
                row[header] = cell.get_text(strip=True)

        rows.append(row)

    # Affichage
    for r in rows[:5]:  # juste un aperçu
        print(r)

    print(f"\nTotal lignes : {len(rows)}")


Tableau dataTable introuvable.


In [10]:
#url = "https://fftri.t2area.com/calendrier/triathlon-de-baudreix/triathlon-l/2023.html"
url = "https://fftri.t2area.com/calendrier/bayman/triathlon-xxl/2026.html"
url = "https://fftri.t2area.com/calendrier/triathlon-de-belfort/triathlon-l-clm/2007.html"
url = "https://fftri.t2area.com/calendrier/aquathlon-de-privas/aquathlon-s/2022.html"

# En-têtes pour simuler un vrai navigateur et éviter les blocages de type 403
headers = {
  "User-Agent": (
      "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML,"
      " like Gecko) Chrome/120.0.0.0 Safari/537.36"
  )
}

print(f"Requête en cours vers : {url}")
response = requests.get(url, headers=headers)

soup = BeautifulSoup(response.text, "html.parser")
results_data = []

# Recherche de tous les blocs de résultats basés sur la classe exacte fournie
result_blocks_a = soup.find_all("a", class_="edition-result-main")
result_blocks_div = soup.find_all("div", class_="edition-result-main")

Requête en cours vers : https://fftri.t2area.com/calendrier/aquathlon-de-privas/aquathlon-s/2022.html


In [16]:
for div_detail in soup.find_all("div", class_="note__content"):
    p = div_detail.find("p")
    if p:
        p_note = p.get_text(strip=True)
print(p_note)

Edition introuvable, nous vous redirigeons vers le calendrier


In [15]:
len(result_blocks_a), len(result_blocks_div)

(0, 0)

In [75]:
result_blocks = result_blocks_div
#result_blocks = result_blocks_a
for block in result_blocks:
    # 1. Extraction du classement
    rank_elem = block.find("span", class_="edition-rank")
    rank = rank_elem.get_text(strip=True) if rank_elem else "N/A"

    # 2. Extraction du nom de l'athlète
    name_elem = block.find("div", class_="edition-name")
    name = name_elem.get_text(strip=True) if name_elem else "N/A"

    # 3. Extraction du club et de la catégorie
    club_elem = block.find("div", class_="edition-club")
    club = club_elem.get_text(separator=" ", strip=True) if club_elem else "N/A"

    # 4. Extraction du temps total
    time_elem = block.find("span", class_="edition-time")
    total_time = time_elem.get_text(strip=True) if time_elem else "N/A"

    # 5. Récupération du lien individuel de l'athlète
    profile_link = block.get("href", "N/A")

    results_data.append({
        "Classement": rank,
        "Athlète": name,
        "Club / Infos": club,
        "Temps Total": total_time,
        "Lien Fiche": profile_link,
    })
    break

In [76]:
len(results_data)

1

In [104]:
results_data

[{'Classement': '1',
  'Athlète': 'DELMAS Baptiste',
  'Club / Infos': 'A.S. MONACO TRIATHLON · MS1',
  'Temps Total': '08:30:51',
  'Lien Fiche': 'N/A'}]

In [109]:
block.find("span", class_="edition-cat-badge").get_text(strip=True)

'MS1'

In [113]:
block.find("div", class_="edition-club").contents[0].strip()

'A.S. MONACO TRIATHLON ·'

In [91]:
block_parent = block.find_parent()
div_splits = block_parent.find_all("div", class_="edition-split")
details_link = block_parent.find_all("a", class_="edition-details-link")

In [94]:
details_link

[<a class="edition-details-link" href="https://fftri.t2area.com/calendrier/bayman/triathlon-xxl/2026/B67354.html">
 <i aria-hidden="true" class="fa-solid fa-list-ol"></i> Voir le résultat
                                                 </a>,
 <a class="edition-details-link" href="/athletes/b67354.html">
 <i aria-hidden="true" class="fa-solid fa-user"></i> Voir l'athlète
                                                     </a>,
 <a class="edition-details-link" href="/clubs/as-monaco-triathlon.html">
 <i aria-hidden="true" class="fa-solid fa-people-group"></i> Voir le club
                                                 </a>]

In [100]:
for element in div_splits:
    label_elem = element.find("small")
    label = label_elem.get_text(strip=True) if label_elem else ""
    time_elem = element.find("b")
    time_value = time_elem.get_text(strip=True) if time_elem else ""
    print(label,time_value)

for element in details_link:
    if element.find("i", class_="fa-list-ol"):
        href = element.get("href")
        print(href)

Natation 00:56:20
T1 00:02:07
Vélo 04:43:32
T2 00:02:47
Course à Pied 02:46:05
https://fftri.t2area.com/calendrier/bayman/triathlon-xxl/2026/B67354.html


In [34]:
# récupération des informations d'un participant

In [12]:
url = "https://fftri.t2area.com/calendrier/triathlon-de-lusigny/triathlon-l/2007.html"
resp = requests.get(url)
resp.raise_for_status()
html = resp.text

m = re.search(r"édition du\s*([0-9]{2}-[0-9]{2}-[0-9]{4})", html)
if m:
    date_str = m.group(1)
    print(date_str)
    date_obj = datetime.strptime(date_str, "%d-%m-%Y")
    print(date_obj.year)

03-06-2007
2007


In [15]:
#URL = "https://fftri.t2area.com/calendrier/triathlon-de-baudreix/triathlon-l/2023/id-987758.html"
URL = "https://fftri.t2area.com/calendrier/triathlon-de-baudreix/triathlon-l/2015/bib-dossard_161.html"

full_text = []

response = requests.get(URL)
response.raise_for_status()

soup = BeautifulSoup(response.content, "html.parser")

# Trouver la balise <section> et ensuite toutes les balises <ul> à l'intérieur
section = soup.find("section")

if not section:
    print("Aucune balise <section> trouvée.")
else:
    # Récupérer toutes les listes <ul> dans la section
    lists = section.find_all("ul")
    for idx, ul in enumerate(lists, start=1):
        for li in ul.find_all("li"):
            text = li.get_text(strip=True)
            full_text.append(text)

In [16]:
for div in soup.find_all("div", class_="note__content"):
    p = div.find("p")
    if p:
        print(p.get_text(strip=True))

Résultat introuvable, nous vous redirigeons vers le calendrier


In [9]:
full_text

['Share on Twitter',
 'Share on Facebook',
 'Share on Linkedin',
 'Share by Email',
 'Disciplines FFTRIAquathlonBike & RunCross DuathlonCross TriathlonDuathlonDuathlon des neigesEnduroRaidsSwimBikeSwimRunTriathlonTriathlon des neiges',
 'Distances FFTRIXSSMLXLXXLXXSXXS openXS openS openJeunes 1 (6-9)Jeunes 2 (10-13)XXS jeunesXS jeunesS jeunes',
 "Ligues régionalesAUVERGNE RHONE ALPESBOURGOGNE-FRANCHE-COMTÉBRETAGNECENTRE VAL DE LOIRECORSEFÉDÉRATION TAHITIENNEGRAND-ESTGUADELOUPEHAUTS DE FRANCEILE-DE-FRANCEMARTINIQUENORMANDIENOUVELLE CALÉDONIENOUVELLE-AQUITAINEOCCITANIEPAYS DE LA LOIREPROVENCE-ALPES-CÔTE D'AZURRÉUNION",
 'MoisJanvierFévrierMarsAvrilMaiJuinJuilletAoûtSeptembreOctobreNovembreDécembre']

In [56]:
rang_global = full_text[0].replace("Rang Global", "")
rang_sexe = full_text[1].replace("Rang Sexe", "")
rang_categorie = full_text[2].replace("Rang Catégorie", "")
categorie = full_text[3].replace("CATÉGORIE", "")
indice_performance = full_text[4].replace("IP (IPR)", "").split(" ")[0].split("=")[0]
indice_performance_reference = full_text[4].replace("IP (IPR)", "").split(" ")[1].replace("(","").replace(")","")
total_time = full_text[6].replace("Temps", "")

swim_time = full_text[11].replace("Temps", "")
rang_swim = full_text[12].replace("Rang / Natation","")
rang_course_swim = full_text[13].replace("Rang / Course","").split("+")[0].split("=")[0]
indice_performance_swim = full_text[14].replace("IP Natation", "")

t1_time = full_text[16].replace("Temps", "")
rang_t1 = full_text[17].replace("Rang / Transition 1","")
rang_course_t1 = full_text[18].replace("Rang / Course","").split("+")[0].split("=")[0]
indice_performance_t1 = full_text[19].replace("IP Transition 1", "")

bike_time = full_text[21].replace("Temps", "")
rang_bike = full_text[22].replace("Rang / Vélo","")
rang_course_bike = full_text[23].replace("Rang / Course","").split("+")[0].split("=")[0]
indice_performance_bike = full_text[24].replace("IP Vélo", "")

t2_time = full_text[26].replace("Temps", "")
rang_t2 = full_text[27].replace("Rang / Transition 2","")
rang_course_t2 = full_text[28].replace("Rang / Course","").split("+")[0].split("=")[0]
indice_performance_t2 = full_text[29].replace("IP Transition 2", "")

run_time = full_text[31].replace("Temps", "")
rang_run = full_text[32].replace("Rang / Course à Pied","")
rang_course_run = full_text[33].replace("Rang / Course","").split("+")[0].split("=")[0]
indice_performance_run = full_text[34].replace("IP Course à Pied", "")

print(rang_global, rang_sexe, rang_categorie, categorie, indice_performance, indice_performance_reference, total_time, swim_time, rang_swim)
print(rang_course_swim, indice_performance_swim, t1_time, rang_t1, rang_course_t1, indice_performance_t1)
print(bike_time, rang_bike, rang_course_bike, indice_performance_bike)
print(t2_time, rang_t2, rang_course_t2, indice_performance_t2)
print(run_time, rang_run, rang_course_run, indice_performance_run)

1 1 1 MSE 321 0 04:02:53 00:42:39 11
11 105 00:00:00 0  0
01:58:34 1 2 321
00:00:00 0 2 0
01:21:40 3 1 255


In [63]:
def extract_race_name(url):
    path = urlparse(url).path
    parts = path.strip("/").split("/")
    print(parts)
    annee = os.path.splitext(parts[3])[0]
    name = parts[1].replace("-", " ")
    distance = parts[2].replace("-", " ")
    return name, distance, annee

extract_race_name("https://fftri.t2area.com/calendrier/ironman-de-nice/triathlon-l/2022.html")

['calendrier', 'ironman-de-nice', 'triathlon-l', '2022.html']


('ironman de nice', 'triathlon l', '2022')

In [66]:
def create_race_unicity(url):
    path = urlparse(url).path
    parts = path.strip("/").split("/")
    return os.path.splitext("_".join(parts[1:]))[0]

create_race_unicity("https://fftri.t2area.com/calendrier/ironman-de-nice/triathlon-l/2022.html")

'ironman-de-nice_triathlon-l_2022'

In [24]:
page_event = "/".join("https://fftri.t2area.com/calendrier/ironman-de-nice/triathlon-l/2022.html".split("/")[:-1])
print(page_event)

https://fftri.t2area.com/calendrier/ironman-de-nice/triathlon-l


In [25]:
resp = requests.get(page_event)
resp.raise_for_status()

soup = BeautifulSoup(resp.text, "html.parser")

article = soup.find("article", class_="edition-description-content")
if article:
    # texte brut propre
    text = article.get_text(separator="\n", strip=True)
    print(text)
else:
    print("article.edition-description-content non trouvé")

L'édition 2026 de l'épreuve
Half IronMan 70.3 de Nice (06) - Triathlon L
aura lieu en date du 28 Juin 2026 dans la région de NICE (FR). Ce triathlon compte pour le
Challenge National Triathlon Longues Distances
. Une fois l'épreuve passée retrouvez ici toutes nos analyses et nos statistiques détaillées.
Manifestation :
IRONMAN France Nice & IRONMAN 70.3 Nice
L
Mixte
Triathlon
TRI
L
TRI
XL
TRI
XXL


In [27]:
text.split("\n")[1]

'Half IronMan 70.3 de Nice (06) - Triathlon L'

In [48]:
MONTHS_FR = {
    "janvier": 1,
    "février": 2,
    "fevrier": 2,
    "mars": 3,
    "avril": 4,
    "mai": 5,
    "juin": 6,
    "juillet": 7,
    "août": 8,
    "aout": 8,
    "septembre": 9,
    "octobre": 10,
    "novembre": 11,
    "décembre": 12,
    "decembre": 12,
}

def extract_date_from_text(text):
    pattern = r"(\d{1,2}\s+[A-Za-zéèêûôîçÉÈÊÛÔÎÇ]+\s+\d{4})"
    m = re.search(pattern, text)
    if m:
        date_str = m.group(1)
        day, month_fr, year = date_str.lower().split()
        return date(int(year), MONTHS_FR[month_fr], int(day))

text_content = text.split("\n")[2]
print(text_content)

aura lieu en date du 28 Juin 2026 dans la région de NICE (FR). Ce triathlon compte pour le


In [49]:
extract_date_from_text(text_content)

datetime.date(2026, 6, 28)

# Uniformisation des fichiers

In [4]:
protiming = pd.read_csv("./sources/protiming/7984-Triathlon-Annette-K_654.csv")

In [5]:
protiming.head(2)

,Pos.,Temps,Nom - Prénom,Dossard,Catégorie,Natation,Velo,CaP,Unnamed: 8,name_epreuve,format_epreuve,date_epreuve,lieu_epreuve,url_epreuve,year_epreuve
0,1,00:29:33,LUDET Cedric,227,HOM,00:05:58,00:16:28,00:07:07,NaN,Triathlon Annette K.,Triathlon XS,06/12/2025,Paris (75),https://protiming.fr/Results/running/7984-Triathlon-Annette-K/654,2025
1,2,00:31:08,BUFFE Julien,75,HOM,00:06:22,00:17:09,00:07:37,NaN,Triathlon Annette K.,Triathlon XS,06/12/2025,Paris (75),https://protiming.fr/Results/running/7984-Triathlon-Annette-K/654,2025


In [6]:
set(protiming["Catégorie"].to_list())

{'FEM', 'HOM'}

In [7]:
chronoweb = pd.read_csv("./sources/chronoweb/Resultats_Triathlon_dOlron_2025_M.csv")

In [8]:
chronoweb.head(2)

,Clt,Nom,Club,Dos,Temps,Cat,Unnamed: 6,Temps .1,Moy,Clt.1,TR1,Temps .2,Moy.1,Clt.2,FinV,GàV,TR2,Temps .3,Moy.2,Clt.3,GàP,name_epreuve,year_epreuve,format_epreuve,url_epreuve
0,1,Paul DELAMARE,ROUEN TRIATHLON,52,01:59:32,1/MJU,NaN,00:18:56,1:15,1.0,00:05:37,00:57:27,41.7,1.0,1.0,0.0,00:02:38,00:34:52,17.2,2.0,0.0,Triathlon dOlron,2025,M,https://chronoweb.com/files/xls/Resultats_Triathlon_dOlron_2025.xlsx
1,2,Nicolas ONILLON,SHARKS TRI 79,199,02:10:08,1/MVE,NaN,00:22:09,1:28,2.0,00:06:21,01:00:48,39.4,4.0,2.0,0.0,00:02:51,00:37:57,15.8,6.0,0.0,Triathlon dOlron,2025,M,https://chronoweb.com/files/xls/Resultats_Triathlon_dOlron_2025.xlsx


In [9]:
set(chronoweb["Cat"].to_list())

{'1/FSE',
 '1/FVE',
 '1/MJU',
 '1/MSE',
 '1/MVE',
 '10/FSE',
 '10/MSE',
 '10/MVE',
 '100/MSE',
 '101/MSE',
 '102/MSE',
 '103/MSE',
 '104/MSE',
 '105/MSE',
 '106/MSE',
 '107/MSE',
 '108/MSE',
 '109/MSE',
 '11/FSE',
 '11/MSE',
 '11/MVE',
 '110/MSE',
 '111/MSE',
 '112/MSE',
 '113/MSE',
 '114/MSE',
 '115/MSE',
 '116/MSE',
 '117/MSE',
 '118/MSE',
 '119/MSE',
 '12/FSE',
 '12/MSE',
 '12/MVE',
 '120/MSE',
 '121/MSE',
 '122/MSE',
 '123/MSE',
 '124/MSE',
 '125/MSE',
 '126/MSE',
 '127/MSE',
 '128/MSE',
 '129/MSE',
 '13/FSE',
 '13/MSE',
 '13/MVE',
 '130/MSE',
 '131/MSE',
 '132/MSE',
 '133/MSE',
 '134/MSE',
 '135/MSE',
 '136/MSE',
 '137/MSE',
 '138/MSE',
 '139/MSE',
 '14/FSE',
 '14/MSE',
 '14/MVE',
 '140/MSE',
 '141/MSE',
 '142/MSE',
 '143/MSE',
 '144/MSE',
 '145/MSE',
 '146/MSE',
 '147/MSE',
 '148/MSE',
 '149/MSE',
 '15/FSE',
 '15/MSE',
 '15/MVE',
 '150/MSE',
 '151/MSE',
 '152/MSE',
 '153/MSE',
 '154/MSE',
 '155/MSE',
 '156/MSE',
 '157/MSE',
 '158/MSE',
 '159/MSE',
 '16/FSE',
 '16/MSE',
 '16/MVE',

In [45]:
prolivesport = pd.read_csv("./sources/prolivesport/1082_AudenciaTriathlonLaBaule2025_M.csv")

In [46]:
prolivesport.head(2)

,Clt,N°,Athlète,Nat.,Club,Temps,Sexe,Cat,Swim,T1,Bike,T2,Run,Bike Start,Bike End,Run Start,Unnamed: 16,name_epreuve,format_epreuve,distance_epreuve,date_epreuve,lieu_epreuve,url_epreuve
0,1,66,VOTTIER Antoine,NaN,POISSY TRIATHLON,01:54:23,M (1e),Senior (1e),00:19:03 (3e),00:02:06 (6e),01:00:02 (2e),00:02:03 (11e),00:31:11 (2e),00:21:09 (3e),01:21:10 (2e),01:23:12 (3e),NaN,Audencia Triathlon La Baule 2025,M,51.5 Km,20/09/2025,La BauleLoire-Atlantique / Pays-de-la-Loire,https://www.prolivesport.fr/index.php?chap=event&sub=liveV3&eventId=1082&race=M
1,2,693,RAPHAEL Aurelien,NaN,ISSY TRIATHLON,01:55:16,M (2e),Senior (2e),00:19:01 (2e),00:01:57 (4e),01:00:12 (4e),00:01:53 (4e),00:32:15 (8e),00:20:57 (1e),01:21:09 (1e),01:23:01 (1e),NaN,Audencia Triathlon La Baule 2025,M,51.5 Km,20/09/2025,La BauleLoire-Atlantique / Pays-de-la-Loire,https://www.prolivesport.fr/index.php?chap=event&sub=liveV3&eventId=1082&race=M


In [61]:
set(prolivesport["Cat"].to_list())

{'Junior  (10e)',
 'Junior  (11e)',
 'Junior  (12e)',
 'Junior  (13e)',
 'Junior  (14e)',
 'Junior  (15e)',
 'Junior  (16e)',
 'Junior  (17e)',
 'Junior  (18e)',
 'Junior  (19e)',
 'Junior  (1e)',
 'Junior  (20e)',
 'Junior  (21e)',
 'Junior  (2e)',
 'Junior  (3e)',
 'Junior  (4e)',
 'Junior  (5e)',
 'Junior  (6e)',
 'Junior  (7e)',
 'Junior  (8e)',
 'Junior  (9e)',
 'Master  (100e)',
 'Master  (101e)',
 'Master  (102e)',
 'Master  (103e)',
 'Master  (104e)',
 'Master  (105e)',
 'Master  (106e)',
 'Master  (107e)',
 'Master  (108e)',
 'Master  (109e)',
 'Master  (10e)',
 'Master  (110e)',
 'Master  (111e)',
 'Master  (112e)',
 'Master  (113e)',
 'Master  (114e)',
 'Master  (115e)',
 'Master  (116e)',
 'Master  (117e)',
 'Master  (118e)',
 'Master  (119e)',
 'Master  (11e)',
 'Master  (120e)',
 'Master  (121e)',
 'Master  (122e)',
 'Master  (123e)',
 'Master  (124e)',
 'Master  (125e)',
 'Master  (126e)',
 'Master  (127e)',
 'Master  (128e)',
 'Master  (129e)',
 'Master  (12e)',
 'Maste

In [78]:
prolivesport["classement_categorie"] = prolivesport["Cat"].str.extract(r"\((\d+)")[0].astype(float)
prolivesport.loc[prolivesport["classement_categorie"] == 0, "classement_categorie"] = np.nan
prolivesport["categorie"] = (
    prolivesport["Cat"]
    .str.replace(r"\s*\(.*\)", "", regex=True)
    .str.strip()
)
prolivesport["natation_classement"] = prolivesport["Swim"].str.extract(r"\((\d+)e\)").astype("Int64")
prolivesport["velo_classement"] = prolivesport["Bike"].str.extract(r"\((\d+)e\)").astype("Int64")

In [4]:
fftrit2area = pd.read_csv("./sources/fftri_t2_area/ironman-de-nice_triathlon-l_2022.csv")

In [8]:
schema = ""
for c in fftrit2area.columns:
    schema += c
    schema += " TEXT,\n"

In [5]:
fftrit2area.head(2)

,Clt,Clt/F,Temps,Nom,Club,CAT,Clt/CAT,id_league,league,Détails,rang_global,rang_sexe,rang_categorie,categorie,indice_performance,indice_performance_reference,total_time,swim_time,rang_swim,rang_course_swim,indice_performance_swim,t1_time,rang_t1,rang_course_t1,indice_performance_t1,bike_time,rang_bike,rang_course_bike,indice_performance_bike,t2_time,rang_t2,rang_course_t2,indice_performance_t2,run_time,rang_run,rang_course_run,indice_performance_run,name_epreuve,format_epreuve,year_epreuve,date_epreuve
0,1039,109.0,06:35:06,Aalya Bux,NaN,F30-34,32.0,28.0,NaN,https://fftri.t2area.com/calendrier/ironman-de-nice/triathlon-l/2022/bib-3995.html,1039,111,32,F30-34,37,0,06:35:06,00:34:14,411,411,75,00:06:17,916,377,41,03:44:25,1178,1087-710,27,00:04:50,247,1042,86,02:05:21,1060,1039,35,Half IronMan 70.3 de Nice (06) - Triathlon L,triathlon l,2026,2026-06-28
1,762,NaN,06:12:45,Abdelkader Sakhri,TRIATHLON SANNOIS FRANCONVILLE,M40-44,93.0,9.0,ILE-DE-FRANCE,https://fftri.t2area.com/calendrier/ironman-de-nice/triathlon-l/2022/bib-3860.html,762,696,93,M40-44,54,42,06:12:45,00:31:49,171,171,90,00:08:10,1391,328-157,11,03:21:40,719,622-294,56,00:08:35,1307,683-61,20,02:02:34,979,762-79,40,Half IronMan 70.3 de Nice (06) - Triathlon L,triathlon l,2026,2026-06-28


In [40]:
set(fftrit2area["CAT"].to_list())

{'F18-24',
 'F25-29',
 'F30-34',
 'F35-39',
 'F40-44',
 'F45-49',
 'F50-54',
 'F55-59',
 'F60-64',
 'M18-24',
 'M25-29',
 'M30-34',
 'M35-39',
 'M40-44',
 'M45-49',
 'M50-54',
 'M55-59',
 'M60-64',
 'M65-69',
 'M70-74'}

In [97]:
TARGET_COLUMNS = [
    "rang_global",
    "temps_total",
    "nom",
    "prenom",
    "dossard",
    "club",
    "categorie",
    "classement_categorie",
    "sexe",
    "classement_sexe",
    "natation",
    "natation_classement",
    "t1",
    "t1_classement",
    "velo",
    "velo_classement",
    "t2",
    "t2_classement",
    "cap",
    "cap_classement",
    "name_epreuve",
    "format_epreuve",
    "distance_epreuve",
    "type_epreuve",
    "date_epreuve",
    "lieu_epreuve",
    "url_epreuve",
    "year_epreuve",
    "source"
]

In [98]:
PROTIMING_MAP = {
    "Pos.": "rang_global",
    "Temps": "temps_total",
    "Nom - Prénom": "nom_prenom",
    "Dossard": "dossard",
    "Catégorie": "categorie",
    "Natation": "natation",
    "Velo": "velo",
    "CaP": "cap",
    "name_epreuve": "name_epreuve",
    "format_epreuve": "format_epreuve",
    "date_epreuve": "date_epreuve",
    "lieu_epreuve": "lieu_epreuve",
    "url_epreuve": "url_epreuve",
    "year_epreuve": "year_epreuve"
}

In [99]:
PROLIVESPORT_MAP = {
    "Clt": "rang_global",
    "Temps": "temps_total",
    "Athlète": "nom_prenom",
    "N°": "dossard",
    "Club": "club",
    "Sexe": "sexe",
    "Cat": "categorie",
    "Swim": "natation",
    "T1": "t1",
    "Bike": "velo",
    "T2": "t2",
    "Run": "cap",
    "name_epreuve": "name_epreuve",
    "format_epreuve": "format_epreuve",
    "distance_epreuve": "distance_epreuve",
    "date_epreuve": "date_epreuve",
    "lieu_epreuve": "lieu_epreuve",
    "url_epreuve": "url_epreuve"
}

In [100]:
FFTRI_MAP = {
    "rang_global": "rang_global",
    "total_time": "temps_total",
    "Nom": "nom",
    "Club": "club",
    "categorie": "categorie",
    "Clt/CAT": "classement_categorie",
    "Clt/F": "classement_sexe",
    "swim_time": "natation",
    "rang_swim": "natation_classement",
    "t1_time": "t1",
    "rang_t1": "t1_classement",
    "bike_time": "velo",
    "rang_bike": "bike_classement",
    "t2_time": "t2",
    "rang_t2": "t2_classement",
    "run_time": "cap",
    "rang_run": "cap_classement",
    "name_epreuve": "name_epreuve",
    "format_epreuve": "format_epreuve",
    "date_epreuve": "date_epreuve",
    "year_epreuve": "year_epreuve"
}

In [101]:
CHRONOWEB_MAP = {
    "Clt": "rang_global",
    "Nom": "nom",
    "Club": "club",
    "Dos": "dossard",
    "Temps ": "temps_total",
    "Cat": "categorie",
    "Temps .1": "natation",
    "TR1": "t1",
    "Temps .2": "velo",
    "TR2": "t2",
    "Temps .3": "cap",
    "name_epreuve": "name_epreuve",
    "format_epreuve": "format_epreuve",
    "year_epreuve": "year_epreuve",
    "url_epreuve": "url_epreuve"
}

In [102]:
def normalize_csv(path, column_map, source_name):
    df = pd.read_csv(path)
    # Drop colonnes pandas parasites
    df = df.loc[:, ~df.columns.str.contains("^Unnamed")]

    # Rename
    df = df.rename(columns=column_map)

    # Supprime les doublons de colonnes après rename
    df = df.loc[:, ~df.columns.duplicated()]

    # Colonnes cibles
    for col in TARGET_COLUMNS:
        if col not in df.columns:
            df[col] = np.nan

    # Nom / prénom
    if "nom_prenom" in df.columns:
        split = df["nom_prenom"].astype(str).str.split(" ", n=1, expand=True)
        df["nom"] = split[0]
        df["prenom"] = split[1]

    if source_name == "protiming":
        df["sexe"] = df["categorie"].map({"HOM": "M", "FEM": "F"})
        df["categorie"] = np.nan
        
        df["type_epreuve"] = df["format_epreuve"].str.rsplit(" ", n=1).str[0].str.lower()
        df["format_epreuve"] = df["format_epreuve"].str.rsplit(" ", n=1).str[1]

    if source_name == "prolivesport":
        df["classement_categorie"] = df["categorie"].str.extract(r"\((\d+)")[0].astype(float)
        df.loc[df["classement_categorie"] == 0, "classement_categorie"] = np.nan
        df["categorie"] = (df["categorie"].str.replace(r"\s*\(.*\)", "", regex=True).str.strip())
        df["classement_sexe"] = df["sexe"].str.extract(r"\((\d+)")[0].astype(float)
        df.loc[df["classement_sexe"] == 0, "classement_sexe"] = np.nan
        df["sexe"] = (df["sexe"].str.replace(r"\s*\(.*\)", "", regex=True).str.strip())

        df["type_epreuve"] = (df["name_epreuve"].str.lower().str.extract(r"(triathlon|duathlon)", expand=False))

        df["year_epreuve"] = pd.to_datetime(df["date_epreuve"],format="%d/%m/%Y",errors="coerce").dt.year

        df["natation_classement"] = df["natation"].str.extract(r"\((\d+)e\)").astype("Int64")
        df["t1_classement"] = df["t1"].str.extract(r"\((\d+)e\)").astype("Int64")
        df["velo_classement"] = df["velo"].str.extract(r"\((\d+)e\)").astype("Int64")
        df["t2_classement"] = df["t2"].str.extract(r"\((\d+)e\)").astype("Int64")
        df["cap_classement"] = df["cap"].str.extract(r"\((\d+)e\)").astype("Int64")

    if source_name == "fftri":
        df["sexe"] = df["CAT"].str[0]
        split = df["nom"].astype(str).str.split(" ", n=1, expand=True)
        df["prenom"] = split[0]
        df["nom"] = split[1]
        
        df["type_epreuve"] = df["format_epreuve"].str.rsplit(" ", n=1).str[0].str.lower()
        df["format_epreuve"] = df["format_epreuve"].str.rsplit(" ", n=1).str[1].str.upper()

    if source_name == "chronoweb":
        split = df["nom"].astype(str).str.split(" ", n=1, expand=True)
        df["prenom"] = split[0]
        df["nom"] = split[1]

        tmp = df["categorie"].str.split("/", expand=True)
        df["classement_categorie"] = pd.to_numeric(tmp[0], errors="coerce").astype("Int64")
        df["sexe"] = tmp[1].str[0]
        df["categorie"] = tmp[1].str[1:]

        df["type_epreuve"] = (df["name_epreuve"].str.lower().str.extract(r"(triathlon|duathlon)", expand=False))

    df["source"] = source_name

    return df[TARGET_COLUMNS].head(3)


In [103]:
dfs = [
    normalize_csv("./sources/protiming/7984-Triathlon-Annette-K_654.csv", PROTIMING_MAP, "protiming"),
    normalize_csv("./sources/prolivesport/1082_AudenciaTriathlonLaBaule2025_M.csv", PROLIVESPORT_MAP, "prolivesport"),
    normalize_csv("./sources/fftri_t2_area/ironman-de-nice_triathlon-l_2022.csv", FFTRI_MAP, "fftri"),
    normalize_csv("./sources/chronoweb/Resultats_Triathlon_dOlron_2025_M.csv", CHRONOWEB_MAP, "chronoweb"),
]

In [104]:
df_all = pd.concat(dfs, ignore_index=True)

/var/folders/hm/d934n_zn2r14w19nr22fmj_00000gn/T/ipykernel_2785/3890152543.py:1: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_all = pd.concat(dfs, ignore_index=True)


In [105]:
df_all

,rang_global,temps_total,nom,prenom,dossard,club,categorie,classement_categorie,sexe,classement_sexe,natation,natation_classement,t1,t1_classement,velo,velo_classement,t2,t2_classement,cap,cap_classement,name_epreuve,format_epreuve,distance_epreuve,type_epreuve,date_epreuve,lieu_epreuve,url_epreuve,year_epreuve,source
0,1,00:29:33,LUDET,Cedric,227.0,NaN,NaN,<NA>,M,NaN,00:05:58,<NA>,NaN,<NA>,00:16:28,<NA>,NaN,<NA>,00:07:07,<NA>,Triathlon Annette K.,XS,NaN,triathlon,06/12/2025,Paris (75),https://protiming.fr/Results/running/7984-Triathlon-Annette-K/654,2025,protiming
1,2,00:31:08,BUFFE,Julien,75.0,NaN,NaN,<NA>,M,NaN,00:06:22,<NA>,NaN,<NA>,00:17:09,<NA>,NaN,<NA>,00:07:37,<NA>,Triathlon Annette K.,XS,NaN,triathlon,06/12/2025,Paris (75),https://protiming.fr/Results/running/7984-Triathlon-Annette-K/654,2025,protiming
2,3,00:31:52,DE,Poorter,163.0,NaN,NaN,<NA>,M,NaN,00:05:26,<NA>,NaN,<NA>,00:18:30,<NA>,NaN,<NA>,00:07:56,<NA>,Triathlon Annette K.,XS,NaN,triathlon,06/12/2025,Paris (75),https://protiming.fr/Results/running/7984-Triathlon-Annette-K/654,2025,protiming
3,1,01:54:23,VOTTIER,Antoine,66.0,POISSY TRIATHLON,Senior,1.0,M,1.0,00:19:03 (3e),3,00:02:06 (6e),6,01:00:02 (2e),2,00:02:03 (11e),11,00:31:11 (2e),2,Audencia Triathlon La Baule 2025,M,51.5 Km,triathlon,20/09/2025,La BauleLoire-Atlantique / Pays-de-la-Loire,https://www.prolivesport.fr/index.php?chap=event&sub=liveV3&eventId=1082&race=M,2025,prolivesport
4,2,01:55:16,RAPHAEL,Aurelien,693.0,ISSY TRIATHLON,Senior,2.0,M,2.0,00:19:01 (2e),2,00:01:57 (4e),4,01:00:12 (4e),4,00:01:53 (4e),4,00:32:15 (8e),8,Audencia Triathlon La Baule 2025,M,51.5 Km,triathlon,20/09/2025,La BauleLoire-Atlantique / Pays-de-la-Loire,https://www.prolivesport.fr/index.php?chap=event&sub=liveV3&eventId=1082&race=M,2025,prolivesport
5,3,01:56:05,CORMIER,Malo,983.0,BEST TRIATHLON SAINT NAZAIRE,Junior,1.0,M,3.0,00:18:56 (1e),1,00:02:11 (11e),11,01:00:04 (3e),3,00:01:57 (5e),5,00:32:58 (12e),12,Audencia Triathlon La Baule 2025,M,51.5 Km,triathlon,20/09/2025,La BauleLoire-Atlantique / Pays-de-la-Loire,https://www.prolivesport.fr/index.php?chap=event&sub=liveV3&eventId=1082&race=M,2025,prolivesport
6,1039,06:35:06,Bux,Aalya,NaN,NaN,F30-34,32.0,F,109.0,00:34:14,411,00:06:17,916,03:44:25,<NA>,00:04:50,247,02:05:21,1060,Half IronMan 70.3 de Nice (06) - Triathlon L,L,NaN,triathlon,2026-06-28,NaN,NaN,2026,fftri
7,762,06:12:45,Sakhri,Abdelkader,NaN,TRIATHLON SANNOIS FRANCONVILLE,M40-44,93.0,M,NaN,00:31:49,171,00:08:10,1391,03:21:40,<NA>,00:08:35,1307,02:02:34,979,Half IronMan 70.3 de Nice (06) - Triathlon L,L,NaN,triathlon,2026-06-28,NaN,NaN,2026,fftri
8,1269,06:58:10,KAMALI,Abdessamad,NaN,NaN,M40-44,141.0,M,NaN,00:38:15,884,00:06:28,969,03:40:32,<NA>,00:07:30,1074,02:25:26,1472,Half IronMan 70.3 de Nice (06) - Triathlon L,L,NaN,triathlon,2026-06-28,NaN,NaN,2026,fftri
9,1,01:59:32,DELAMARE,Paul,52.0,ROUEN TRIATHLON,JU,1.0,M,NaN,00:18:56,<NA>,00:05:37,<NA>,00:57:27,<NA>,00:02:38,<NA>,00:34:52,<NA>,Triathlon dOlron,M,NaN,triathlon,NaN,NaN,https://chronoweb.com/files/xls/Resultats_Triathlon_dOlron_2025.xlsx,2025,chronoweb


# Prolisport schema of dataframes

In [7]:
from pathlib import Path

In [8]:
target = "prolivesport"
root = Path(f"./sources/{target}/")
all_files = []
for i in [f'./{p}' for p in root.rglob("*") if p.is_file()]:
    if i != f"./sources/{target}/urls.csv" and i != f"./sources/{target}/.DS_Store":
        all_files.append(i)

In [9]:
# for each file extract columns to know schema of df and filename path
schema_list = []
for local_path in all_files:
    df = pd.read_csv(local_path)
    unnamed_cols = list(df.columns[df.columns.str.contains("Unnamed")])
    for index, u_c in enumerate(unnamed_cols):
        df = df.rename(columns={u_c:"Unnamed_"+str(index)})
    schema_list.append({"local_path": local_path, "columns": list(df.columns)})

In [10]:
def known_columns(col_list, dict_list):
    for index, value in enumerate(dict_list):
        if value["columns"] == col_list:
            return True, index
    return False, -1

In [11]:
# group filename path together when sharing the same schema
merged_info = [{"columns" : schema_list[0]["columns"], "local_paths" : [schema_list[0]["local_path"]]}]
for df_info in schema_list[1:]:
    is_present, index = known_columns(df_info["columns"], merged_info)
    if is_present:
        merged_info[index]["local_paths"].append(df_info["local_path"])
    else:
        merged_info.append({"columns": df_info["columns"], "local_paths": [df_info["local_path"]]})

In [17]:
merged_info

[{'columns': ['Clt',
   'N°',
   'Athlète',
   'Nat_',
   'Club',
   'Temps',
   'Sexe',
   'Cat',
   'Unnamed_0',
   'name_epreuve',
   'format_epreuve',
   'distance_epreuve',
   'date_epreuve',
   'lieu_epreuve',
   'url_epreuve'],
  'local_paths': ['./sources/prolivesport/Bike__Run_Wasquehal_2024_12-1520ans.csv',
   './sources/prolivesport/Bike__Run_Halluin_2024_XS.csv',
   './sources/prolivesport/Swimrun_de_lAa_S.csv',
   './sources/prolivesport/Triathlon__Duathlon_du_Lac_dArdres_2025_Jeunes206-720ans.csv',
   './sources/prolivesport/Bike__Run_Halluin_2024_Benjamin-Minime.csv',
   './sources/prolivesport/Audencia_Triathlon_La_Baule_2025_BE-MI.csv',
   './sources/prolivesport/Bike__Run_Gravelines_2024_Mini-Kid.csv',
   './sources/prolivesport/Bike__Run_Gravelines_2024_Maxi-Kid.csv',
   './sources/prolivesport/TriLong_-_TriaCourt_2025_Triacross20Pupilles.csv',
   './sources/prolivesport/TriaCross_2025_Triacross20Pupilles.csv',
   './sources/prolivesport/Bike__Run_Wasquehal_2024_6-11

In [70]:
def get_example_file(directory):
    path = Path(directory)
    file_list = [p for p in path.iterdir() if not p.is_dir()]
    return "./"+str(file_list[0])

def create_schema_from_file(table_name, example_file_path):
    df = pd.read_csv(example_file_path)
    schema = f"CREATE TABLE raw_{table_name} (\n id SERIAL PRIMARY KEY,"
    for c in df.columns:
        schema += c.replace("/","_").replace(".", "")
        schema += " TEXT,\n"
    schema += "scraped_at TIMESTAMP DEFAULT NOW() );"
    return schema

In [71]:
sources = [{"name": "fftri_t2_area", "example_file": "./sources/fftri_t2_area/bayman_triathlon-xxl_2025.csv"}]
path_prolivesport = Path("./sources/prolivesport")
directories_prolivesport = [p.name for p in path_prolivesport.iterdir() if p.is_dir()]
for d in directories_prolivesport:
    sources.append({"name": d, "example_file": get_example_file("./sources/prolivesport/"+d)})

In [72]:
sources

[{'name': 'fftri_t2_area',
  'example_file': './sources/fftri_t2_area/bayman_triathlon-xxl_2025.csv'},
 {'name': 'prolivesport_51',
  'example_file': './sources/prolivesport/prolivesport_51/Triathlon__Duathlon_du_Lac_dArdres_2025_Triathlon20Relais.csv'},
 {'name': 'prolivesport_56',
  'example_file': './sources/prolivesport/prolivesport_56/Aquathlon_du_calaisis_2021_XXS.csv'},
 {'name': 'prolivesport_60',
  'example_file': './sources/prolivesport/prolivesport_60/Championnats_de_France_de_Duathlon_Individuel_S_-_NoyonSeniors_Masters_Femmes_FEMME.csv'},
 {'name': 'prolivesport_58',
  'example_file': './sources/prolivesport/prolivesport_58/Triathlons_Open_de_Quiberon_2025_M.csv'},
 {'name': 'prolivesport_33',
  'example_file': './sources/prolivesport/prolivesport_33/Audencia_Triathlon_La_Baule_2025_S_Light.csv'},
 {'name': 'prolivesport_34',
  'example_file': './sources/prolivesport/prolivesport_34/Triathlon__Duathlon_de_laudomarois_2021_Triathlon20XS.csv'},
 {'name': 'prolivesport_59',
 

In [9]:
df = pd.read_csv("./sources/chronoweb/Resultats_Levezou_2018_L.csv")

In [10]:
df

,Clt,Noms,Equipe,Dos,Temps,Clt.1,/,Cat,Ecart/1er,Temps .1,Moy/100,Clt.2,TR1,Temps .2,Moy,Clt.3,FinV,GàV,TR2,Temps .3,Moy.1,Clt.4,GàP,name_epreuve,year_epreuve,format_epreuve,url_epreuve
0,1,SAVE Jérôme,L UNION TRIATHLON 31,153,04:06:19.722000,1.0,/,MV1,NaN,00:32:41.276000,00:01:38.064000,2.0,00:01:06.724000,02:07:51.712000,37.540512,1.0,1.0,1.0,00:01:22.288000,01:23:17.722000,14.406564,3.0,0.0,Levezou,2018,L,https://chronoweb.com/files/xls/Resultats_Levezou_2018.xlsx
1,2,DOUMENC Sacha,TRIATHLON CLUB CARCASSONNAIS,141,04:10:07.614000,1.0,/,MS2,00:03:47.892000,00:32:40.375000,00:01:38.019000,1.0,00:01:32.625000,02:11:29.516000,36.504140,3.0,2.0,-1.0,00:00:57.484000,01:23:27.614000,14.378105,4.0,0.0,Levezou,2018,L,https://chronoweb.com/files/xls/Resultats_Levezou_2018.xlsx
2,3,LAFON Jean-baptiste,A. L. LAYRAC TRIATHLON,82,04:11:38.846000,2.0,/,MS2,00:05:19.124000,00:32:52.636000,00:01:38.632000,4.0,00:01:49.364000,02:11:02.514000,36.629506,2.0,3.0,1.0,00:01:08.486000,01:24:45.846000,14.156937,6.0,0.0,Levezou,2018,L,https://chronoweb.com/files/xls/Resultats_Levezou_2018.xlsx
3,4,LABADENS Cédric,NaN,115,04:16:06.210000,1.0,/,MS3,00:09:46.488000,00:34:21.180000,00:01:43.059000,5.0,00:01:22.820000,02:14:52.091000,35.590307,6.0,5.0,0.0,00:01:30.909000,01:23:59.210000,14.287954,5.0,1.0,Levezou,2018,L,https://chronoweb.com/files/xls/Resultats_Levezou_2018.xlsx
4,5,SIMON Xavier,BRIVE LIMOUSIN TRIATHLON,66,04:16:33.592000,2.0,/,MV1,00:10:13.870000,00:32:45.315000,00:01:38.266000,3.0,00:01:12.685000,02:19:28.421000,34.415095,11.0,7.0,-4.0,00:01:24.579000,01:21:42.592000,14.686109,1.0,2.0,Levezou,2018,L,https://chronoweb.com/files/xls/Resultats_Levezou_2018.xlsx
5,6,HALLEUMIEUX Christophe,TEAM MERMILLOD TRIATHLON,158,04:16:45.526000,1.0,/,MV2,00:10:25.804000,00:36:36.244000,00:01:49.812000,22.0,00:01:51.756000,02:15:04.349000,35.536476,7.0,8.0,14.0,00:01:07.651000,01:22:05.526000,14.617728,2.0,2.0,Levezou,2018,L,https://chronoweb.com/files/xls/Resultats_Levezou_2018.xlsx
6,7,DELAMOTTE Florent,TRIATHLON LITTORAL 59,21,04:25:16.432000,1.0,/,MS4,00:18:56.710000,00:35:55.497000,00:01:47.775000,14.0,00:01:29.503000,02:13:05.542000,36.065179,4.0,4.0,10.0,00:01:02.458000,01:33:43.432000,12.803569,19.0,-3.0,Levezou,2018,L,https://chronoweb.com/files/xls/Resultats_Levezou_2018.xlsx
7,8,DELOGE Guillaume,TRINOSAURES,14,04:25:20.524000,2.0,/,MS4,00:19:00.802000,00:36:36.943000,00:01:49.847000,23.0,00:01:28.057000,02:16:46.922000,35.092328,9.0,9.0,14.0,00:01:36.078000,01:28:52.524000,13.502049,10.0,1.0,Levezou,2018,L,https://chronoweb.com/files/xls/Resultats_Levezou_2018.xlsx
8,9,MAGANA Frédéric,ALBI TRIATHLON,131,04:25:25.414000,2.0,/,MV2,00:19:05.692000,00:34:26.550000,00:01:43.327000,6.0,00:02:23.450000,02:19:10.310000,34.489738,10.0,10.0,-4.0,00:01:04.690000,01:28:20.414000,13.583845,9.0,1.0,Levezou,2018,L,https://chronoweb.com/files/xls/Resultats_Levezou_2018.xlsx
9,10,PERRIER Nicolas,ASPH TRIATHLON,122,04:27:51.118000,3.0,/,MV2,00:21:31.396000,00:35:28.449000,00:01:46.422000,12.0,00:01:14.551000,02:20:27.389000,34.174286,12.0,11.0,1.0,00:01:21.611000,01:29:19.118000,13.435047,11.0,1.0,Levezou,2018,L,https://chronoweb.com/files/xls/Resultats_Levezou_2018.xlsx
